# Libraries

In [ ]:
import sctop as top
import pandas as pd
import numpy as np
import scanpy as sc
import scipy
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
from collections import Counter
import plotly.io as pio
import anndata as ad
import h5py
import sys
import os
os.chdir('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Differentiation/scripts')
sys.path.append('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Vilker_Helper_Files/scTOP')
%load_ext autoreload
%autoreload 1
%aimport SimilarityHelper
%aimport TopObject
%aimport CriticalityHelper
%aimport Perturbation
pio.renderers.default = 'notebook'

In [ ]:
### Not necessary now that file has been directly downloaded, but could be needed if update occurs without new download
# basisSpecies, targetSpecies = ("human", "mouse")
# ensemblMart, ensemblConfig, server = TopObject.getEnsemblMart(speciesNames=[basisSpecies, targetSpecies])
# mapping = TopObject.getOrthologMapping(basisSpecies, targetSpecies, ensemblMart, ensemblConfig)

## Add Dataset

In [ ]:
## Add dataset with prompts to guide you
summaryFile = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Vilker_Helper_Files/scTOP/DatasetInformation.csv"
TopObject.dynamicAddDataset(summaryFile=summaryFile)

# Negretti

In [ ]:
negretti = TopObject.TopObject("Negretti", skipProcess=True, keep=True)
# negretti.anndata

In [ ]:
# negretti.metadata["timepoint"].value_counts()
negretti.annotations.value_counts()

In [ ]:
negretti.project(simplifiedMouseBasis, "MC-KO", alignGenes=True, normalize=False)

In [ ]:
includeCriteria = None
negrettiSimilarityMap = SimilarityHelper.getMatchingProjections(negretti, "MC-KO", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(negrettiSimilarityMap, 
        title="Negretti vs MC-KO Similarity Boxplot",
        # outFile="../../PendingResults/negretti vs MC-KO (Updated, Smallest) Boxplot.png"
)

# Plasschaert

In [ ]:
plasschaert = TopObject.TopObject("Plasschaert", skipProcess=True)
plasschaert.anndata

In [ ]:
plasschaert.process()

In [ ]:
plasschaert.cellTypeColumn = "clusters_Fig1"
plasschaert.setMetadata()
plasschaert.annotations.value_counts()

In [ ]:
plasschaert.metadata["timepoint"].value_counts()

In [ ]:
# target = "Basal"
# includeCriteria = plasschaert.annotations.isin(["Ciliated", "Secretory", target])
# # includeCriteria = None
# plasschaertDiffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(plasschaert.anndata, plasschaert.cellTypeColumn, target, 
#                         individualCompare=True, includeCriteria=includeCriteria
# )

# includeCriteria = np.logical_and(includeCriteria, plasschaert.annotations == target)
# includeCriteria = bharat.annotations == target
plasschaertTargetDF = CriticalityHelper.getCombinedTopGenes(plasschaertDiffTableMap, plasschaert.processed, 
                        includeCriteria=includeCriteria, missesAllowed=1, minimumChange=0.15, expressionThreshold=0.001, checkSurface=False)
plasschaertTargetDF

In [ ]:
# negrettiPlasschaert = negretti.copy()
negrettiPlasschaert.df

In [ ]:
plasschaert.df.index.name = "gene"
plasschaert.df

In [ ]:
# plasFilter = plasschaert.annotations == "Basal"
# overlap = np.intersect1d(negretti.df.index, plasschaert.df.loc[:, plasFilter].index)
# len(overlap)
# df = pd.merge(negretti.df, plasschaert.df.loc[:, plasFilter], on=negretti.df.index.name, how="inner")
df

In [ ]:
negrettiPlasschaert.metadata.loc[:, ["celltype", "timepoint"]]

In [ ]:
# annotations = list(negretti.annotations) + list(plasschaert.annotations[plasFilter])
# negrettiPlasschaert.anndata = ad.AnnData(df.T)
# negrettiPlasschaert.df = df
# negrettiPlasschaert.anndata.var = pd.DataFrame(df.index, index=df.index)
# obs = pd.DataFrame({"celltype": annotations, "identity": list(negretti.metadata["timepoint"]) + list(plasschaert.metadata.loc[plasFilter, "timepoint"])}, index=df.columns)
# negrettiPlasschaert.anndata.obs = obs
# negrettiPlasschaert.cellTypeColumn = "celltype"
# negrettiPlasschaert.setMetadata()

# negrettiPlasschaert.setAnndata(sc.read_h5ad("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/AnnData/Combined/NegrettiPlasschaert.h5ad"))
# negrettiPlasschaert.filter(maxSamples=500)
# SimilarityHelper.writeAnnData(negrettiPlasschaert.anndata, "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/AnnData/Combined/NegrettiPlasschaert.h5ad")

In [ ]:
genesSelectedFrame, geneProportionFrame = negrettiPlasschaert.getBestGenes()

In [ ]:
geneProportionFrame

In [ ]:
# negrettiPlasschaert.process()
# from sklearn.feature_selection import SelectKBest, f_classif
# selector = SelectKBest(score_func=f_classif, k=int(0.1 * len(negrettiPlasschaert.df.index)))
# trainSelected = selector.fit_transform(negrettiPlasschaert.processed.T, negrettiPlasschaert.annotations)
selectedFeatures = negrettiPlasschaert.df.index[selector.get_support()]
len(selectedFeatures)

In [ ]:
genesSelectedFrame

In [ ]:
goodGenes = genesSelectedFrame[genesSelectedFrame["Successes"] > 4].index
len(goodGenes)

In [ ]:
negrettiPlasschaert.setAnndata(negrettiPlasschaert.anndata[:, negrettiPlasschaert.df.index.isin(list(selectedFeatures))])
negrettiPlasschaert.setBasis()
# negrettiPlasschaert.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/NegrettiPlasschaert.csv", index_label="gene")

In [ ]:
basisCollection = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/BasisCollection.csv"
NP = SimilarityHelper.loadBasis(basisName="NegrettiPlasschaert", basisCollection=basisCollection)

# Montoro

In [ ]:
montoro = TopObject.TopObject("Montoro", skipProcess=True)
montoro.metadata

In [ ]:
montoro.annotations.value_counts()

In [ ]:
montoro.metadata["mouse"].value_counts()

In [ ]:
montoroTurnover = TopObject.TopObject("MontoroTurnover", skipProcess=True)
montoroTurnover.metadata

In [ ]:
montoroTurnover.annotations.value_counts()

In [ ]:
montoroTurnover.project(NM, "Negretti-Montoro")

In [ ]:
montoroSimilarityMap = SimilarityHelper.getMatchingProjections(montoroTurnover, "Negretti-Montoro", includeCriteria=None)
SimilarityHelper.similarityBoxplot(montoroSimilarityMap, 
        title="MontoroTurnover vs Negretti-Montoro Similarity Boxplot",
        # outFile="../../PendingResults/MontoroTurnover vs Negretti-Montoro Boxplot.png"
)

In [ ]:
SimilarityHelper.plotTwoMultiple(
    montoroTurnover, "Negretti-Montoro", "Ciliated", "Basal", 
    unsupervisedContour=False, #gene="TDTOMATO-EXTRA-IVS",
    # alpha=1, markerSize=150, legendMarkerScale=0.25, axisFontSize=32, legendFontSize=32, DPI=300,
    # plotInRow=True, outFile="../../PendingResults/MontoroTurnover vs NM AT1 vs AT2 All Days Unsupervised.png"
)

# Combining Bases

In [ ]:
negrettiFilter = np.logical_and(negretti.metadata["timepoint"].isin(["P7", "P3", "P5", "P14"]), negretti.annotations != "Secretory")
negrettiMontoro = montoro.mergeWithOther(negretti, includeCriteriaOther=negrettiFilter)

In [ ]:
genesSelectedFrame, geneProportionFrame = negrettiMontoro.getBestGenes(proportions=[0.3, 0.4, 0.5, 0.6, 0.7], trialCount=1)
geneProportionFrame

In [ ]:
# NMOG = negrettiMontoro.copy()
negrettiMontoro.setAnndata(NMOG.anndata)
negrettiMontoro.filter(maxSamples=750, skipProcess=False)
negrettiMontoro.filterBestGenes(0.1)

In [ ]:
negrettiFilter = np.logical_and(negretti.metadata["timepoint"].isin(["P7", "P3", "P5", "P14"]), ~negretti.annotations.isin(["Secretory", "Ciliated"]))
montoroFilter = montoroTurnover.annotations.isin(["Basal", "Club", "Ciliated"])
negrettiMontoroTurnover = montoroTurnover.mergeWithOther(negretti, includeCriteriaSelf=montoroFilter, includeCriteriaOther=negrettiFilter)
negrettiMontoroTurnover.annotations.value_counts()

In [ ]:
# NMTOG.filter(maxSamples=15000)
del negrettiMontoroTurnover
negrettiMontoroTurnover = NMTOG.copy()
# # NMTOG = negrettiMontoroTurnover.copy()
# # negrettiMontoroTurnover.setAnndata(NMTOG.anndata)
# negrettiMontoroTurnover.filter(maxSamples=1000)
negrettiMontoroTurnover.filterBestGenes(0.2)
negrettiMontoroTurnover.setBasis()

In [ ]:
NMTOG.annotations.value_counts()

In [ ]:
genesSelectedFrame, geneProportionFrame = NMM.getBestGenes(proportions=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7], trialCount=1)
geneProportionFrame

In [ ]:
NMP.annotations.value_counts()

In [ ]:
# NMMOG = NMM.copy()
NMM = NMOG.mergeWithOther(mcCall, includeCriteriaOther=mcCall.annotations=="Alveolar intermediate")
NMM.filter(maxSamples=3000, skipProcess=True)
NMM.filterBestGenes(proportion=0.1)
NMM.setBasis()

In [ ]:
# NMMOG = NMM.copy()
NMP = negrettiMontoro.mergeWithOther(planer, includeCriteriaOther=planer.annotations=="Alveolar_transitional")
# NMP.filter(maxSamples=300, skipProcess=True)
NMP.filterBestGenes(proportion=0.1)
NMP.setBasis()

In [ ]:
# negrettiMontoro.setAnndata(negrettiMontoro.anndata[:, negrettiMontoro.df.index.isin(list(selectedFeatures))])
# negrettiMontoro.setBasis(allowedGenes=None)
negrettiMontoroTurnover.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/NegrettiMontoroTurnover1000.csv", index_label="gene")

In [ ]:
NMP.annotations.value_counts()

In [ ]:
negrettiMontoro.testBasis()

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(negrettiMontoro)

In [ ]:
pd.read_csv(basisCollection, index_col="Name")

In [ ]:
# NM = negrettiMontoro.basis
basisCollection = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/BasisCollection.csv"
NM = SimilarityHelper.loadBasis(basisName="NegrettiMontoro", basisCollection=basisCollection)
NMFull = SimilarityHelper.loadBasis(basisName="NegrettiMontoroFull", basisCollection=basisCollection)
lungMAP2500 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="LungMAP2500",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory"]
)
HaberMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500ANOVA2", #HaberMAPANOVA1
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)
HaberMAP500 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)
Adams = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Adams500ANOVA2", #Adams500ANOVA2
                                         #basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)
# Natri = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Natri2000ANOVA3"
# )
Natri = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Natri2000ANOVA3Alone").drop("Proliferating", axis=1)
# Kathiriya3000 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Kathiriya3000"
# )
# KathiriyaABI2 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="KathiriyaABI2"
# )
# KathiriyaABI1 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="KathiriyaABI1"
# )
# KathiriyaRelabeled = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="KathiriyaRelabeled"
# )

# MC-KO Basis

In [ ]:
# anno = pd.read_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/MouseAtlas/MCA3.0_data_annotation.csv")
# anno2 = pd.read_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/MouseAtlas/MCA2.0_cell_info.csv")
anno.loc[anno["stage_tissue "] == "Adult-Lung"]["cell_type"].value_counts()
# anno2.loc[anno2["Tissue"] == "Lung"]["Main_annotation"].value_counts()
# anno2["Tissue"].value_counts()

In [ ]:
data_MC20, metadata_MC20 = top.load_basis('MC-KO', minimum_cells = 50)
metadata_MC20.loc[metadata_MC20["Organ"] == "Lung"]
# metadata_MC20["Source"].value_counts()

In [ ]:
cleanedBasis = SimilarityHelper.loadMCKOBasis()
mouseEpithelialKeep = ["AT1", "AT2", "Basal", "Ciliated", "Club"]
simplifiedMouseBasis = cleanedBasis.loc[:, cleanedBasis.columns.isin(mouseEpithelialKeep)]
simplifiedMouseBasis

In [ ]:
# mouseCorr = (1 / simplifiedMouseBasis.shape[0]) * simplifiedMouseBasis.T.dot(simplifiedMouseBasis)
eta =  (1 / simplifiedMouseBasis.shape[0]) * np.linalg.inv(mouseCorr).dot(simplifiedMouseBasis.T)
predictFrame = pd.DataFrame(eta, index=simplifiedMouseBasis.columns, columns=simplifiedMouseBasis.index)
predictFrame.loc["AT2"].sort_values(ascending=False).head(20)

# Riemondy

In [ ]:
riemondy = TopObject.TopObject("Riemondy", skipProcess=True, keep=True)
riemondy.metadata

In [ ]:
riemondy.metadata["new_expt_id"].value_counts()

In [ ]:
plt.hist(riemondy.metadata.loc[:, "gfp_counts"][riemondy.annotations == "Naive Type I"])#, bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
plt.hist(riemondy.projections["Planer"].loc["Krt5", :][riemondy.annotations == "Cell Cycle Arrest Type II"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
# riemondy.filter(keep=["Naive Type II", "Naive Type I", "Basal", "Cell Cycle Arrest Type II"])
# riemondy.setBasis(threshold=50)
# riemondy.combineBases(choi, firstKeep=['Naive Type II', 'Basal', 'Cell Cycle Arrest Type II'], secondKeep=['AT1'], name="Choi")
riemondy.combineBases(NP, firstKeep=['Cell Cycle Arrest Type II'], name="NP")

In [ ]:
riemondy.annotations.value_counts()

In [ ]:
# riemondy.project(planer.basis, "Planer3")
# riemondy.project(planerFilt, "PlanerFilt")
# riemondy.project(NP, "NP")
# riemondy.project(NM, "Negretti-Montoro")
# riemondy.project(strunz.basis, "StrunzFilt")
# riemondy.project(bibek.combinedBases["BibekCombined"], "BibekCombined")
# riemondy.getOrthologs(mapping)
# riemondy.project(HaberMAP, "HaberMAP")
riemondy.project(Natri, "Natri")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
SimilarityHelper.plotBasisCorrelationMatrix(riemondy)

In [ ]:
includeCriteria=riemondy.annotations.isin(["AT1", "Naive Type II", "Alveolar intermediate", "Early intermediate", "Basal", "MCC"])
riemondy.testBasis(maxBasisSamples=240, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(riemondy, title="riemondy Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/riemondy Confusion Matrix Downsampled Basis 240 Test 500 Trials 5.png"
)

In [ ]:
riemondy.setBasis()
riemondy.combineBases(simplifiedMouseBasis, firstKeep=["Cell Cycle Arrest Type II"], name="MC-KO")
# riemondy.combineBases(simplifiedMouseBasis, firstKeep=["Transdifferentiating Type II"], name="MC-KO T")
# riemondy.combinedBases["MC-KO T"]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
SimilarityHelper.plotPredictivity(ax, riemondy, "Cell Cycle Arrest Type II", title="Riemondy Predictivity")
plt.show()

In [ ]:
# includeCriteria = riemondy.metadata["new_expt_id"].isin(["ATII-injured expt.1", "ATII-injured expt.2"])
riemondySimilarityMap = SimilarityHelper.getMatchingProjections(riemondy, "Negretti-Montoro", includeCriteria=None)
SimilarityHelper.similarityBoxplot(riemondySimilarityMap, title="Riemondy vs Negretti-Montoro Similarity Boxplot", 
                                   outFile="../../PendingResults/Riemondy vs NM Boxplot.png"
)

In [ ]:
riemondySimilarityMap = SimilarityHelper.getMatchingProjections(riemondy, "Strunz", includeCriteria=None)
SimilarityHelper.similarityBoxplot(riemondySimilarityMap, title="Riemondy vs Strunz Similarity Boxplot", 
                                   # outFile="../../PendingResults/Riemondy vs Strunz Boxplot.png"
)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(25,20))
# marker_genes = ['Krt8', 'Krt17', 'Krt5', 'Cdkn2a', 'Itga2', 'Palld'] # transition
marker_genes = ['Krt17', 'Scgb1a1', 'Scgb3a2', 'Krt5', 'Krt15', 'Trp63']
# marker_genes = ['Cdk4', 'Krt17', 'Krt5', 'Tgfb2', 'Trp53', 'Palld'] # transition
axs = ax.flatten()
for i, gene in enumerate(marker_genes):
    SimilarityHelper.plotTwo(riemondy, "NP",
             'Basal', 'AT2',
             gene = gene, ax=axs[i], show=False)
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(20,20))
axs = ax.flatten()
for i in range(9):
    SimilarityHelper.plotTwo(riemondy.projections["MC-KO"], riemondy.annotations,
             'AT1', 'AT2', ax=axs[i], seed=i, maxLabelCount=200, unsupervisedContour=True, plotMultiple=True)
plt.savefig("../../PendingResults/Riemondy vs MC-KO AT1 vs AT2 Downsampled x9.png")
plt.show()

In [ ]:
riemondy2.annotations[includeCriteria].value_counts()

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'
includeCriteria = riemondy2.metadata["new_expt_id"].isin(["ATII-injured expt.1", "ATII-injured expt.2"])
SimilarityHelper.plotTwoObj(riemondy2, "Full MC-KO", axis1, axis2, 
         unsupervisedContour=True, axisFontSize=22, legendFontSize=14, markerSize=80,
         includeCriteria=includeCriteria,# maxLabelCount=200,
         title="Riemondy Injury vs MC-KO Murine Basis Similarity Plot", outFile="../../PendingResults/Riemondy Injury vs MC-KO AT1 vs AT2 Unsupervised.png"
)

In [ ]:
axis1 = 'Basal'
axis2 = 'AT2'
smallerKeep = ["Naive Type I", "Naive Type II", "Transdifferentiating Type II", "Injured Type II", "Cell Cycle Arrest Type II", "Basal"] #riemondy.toKeep
includeCriteria = riemondy.annotations.isin(smallerKeep)

ax = SimilarityHelper.plotTwo(riemondy, "Negretti-Montoro", axis1, axis2, 
         unsupervisedContour=False, axisFontSize=22, legendFontSize=14, markerSize=80, legendInner=True, DPI=300,
         # title="Riemondy vs Planer Basal vs AT2", 
         # outFile="../../PendingResults/Riemondy vs NM AT1 vs AT2.png",
         includeCriteria=includeCriteria
)

In [ ]:
import textwrap

fig, ax = plt.subplots(1, 1, figsize=(8,8))
axis1 = 'Basal'
axis2 = 'AT2'
# smallerKeep = riemondy.toKeep.copy()
# smallerKeep.remove("Proliferating Type II")
# smallerKeep.remove("Naive Type II")
includeCriteria = riemondy.annotations.isin(smallerKeep)

SimilarityHelper.plotTwo(riemondy.projections["MC-KO"], riemondy.annotations,
         axis1, axis2, supervisedContour=True, axisFontSize=22, legendFontSize=14, markerSize=80,
         ax=ax, title="Riemondy vs Herriges Murine Basis Similarity Plot", includeCriteria=includeCriteria, outFile='../PendingResults/Riemondy vs MC-KO Basal vs AT2 Supervised (Uncaptioned).png'
)
# caption = "Figure 1. Lung epithelial scRNAseq data of mice injured via LPS from Riemondy et al., plotted against reference basis built from Michael Herriges data in the Kotton Lab."
# wrappedCaption = "\n".join(textwrap.wrap(caption, width=92))

# plt.text(-0.1, -0.17, wrappedCaption, fontsize=13, ha='left', transform=ax.transAxes)

In [ ]:
riemondy.projections["StrunzFilt"]

In [ ]:
axis1 = 'Krt8+ ADI'
# axis1 = "AT1"
axis2 = 'AT2'
# includeCriteria = riemondy.annotations.isin(riemondy.toKeep)
includeCriteria = riemondy.annotations == "Naive Type II"
ax = SimilarityHelper.plotTwo(riemondy, "Strunz", axis1, axis2, 
         unsupervisedContour=False, DPI=100, labels=riemondy.toKeep, labelDimensions=True, legendInner=True, maxLabelCount=None,
         # outFile="../../PendingResults/Riemondy vs Strunz 1000 Krt8+ ADI vs AT2.png",
         includeCriteria=includeCriteria
)

In [ ]:
axis1 = 'KRT5-KRT17+'
# axis1 = "AT1"
axis2 = 'AT2'
# includeCriteria = riemondy.annotations.isin(riemondy.toKeep)
# includeCriteria = riemondy.annotations == "Naive Type II"
ax = SimilarityHelper.plotTwo(riemondy, "Natri", axis1, axis2, 
         unsupervisedContour=False, DPI=100, #labels=riemondy.toKeep, labelDimensions=True, legendInner=True, maxLabelCount=None,
         # outFile="../../PendingResults/Riemondy vs Strunz 1000 Krt8+ ADI vs AT2.png",
         #includeCriteria=includeCriteria
)

In [ ]:
SimilarityHelper.getProjectionStats(riemondy, "Strunz", "Cell Cycle Arrest Type II", 
                                    outFile="../../PendingResults/Riemondy vs Strunz 1000 Stats 1.csv"
)

In [ ]:
SimilarityHelper.getProjectionStatsFocused(riemondy, "Strunz", axis1,
                                            outFile="../../PendingResults/Riemondy vs Strunz 1000 Stats 2.csv"
)

In [ ]:
# riemondy, riemondy_df, riemondy_metadata, riemondyData, riemondyAnnotations, riemondy_kwargs, rimeondyKept
xLabel = 'AT1'
yLabel = 'AT2'
zLabel = 'Krt8 high AT2'
legendTitle = "Riemondy Annotations"
figureTitle = "Riemondy vs MC-KO Basis Similarity Plot"
color = riemondy.annotations
# SimilarityHelper.plotThree(revisedProjections, x, xLabel, y, yLabel, z, zLabel, color)
SimilarityHelper.plotThree(riemondy.projections["BibekCombined"], xLabel, yLabel, zLabel, color, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
CriticalityHelper.stateDistancePlot(riemondy, "MC-KO", includeCriteria=None, quantile=0.5, title="Riemondy vs MC-KO Distances", outFile="Riemondy vs MC-KO Min Euclidean Distance 50% Closest.png")

In [ ]:
riemondy.setBasis()
# cellTypes = ["AT1", "AT2", "Krt8+ ADI", "Ciliated", "Club", "Goblet"]
riemondyDimensionsMap = Perturbation.getSpaceDimensionsAll(riemondy, riemondy.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False, cellTypes=None)
SimilarityHelper.plotSpaceMatrix(riemondyDimensionsMap, outFile="../../PendingResults/Riemondy Perturbation Dimensions.png")

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(riemondy, riemondy.basis, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # outFile="../../PendingResults/Riemondy Perturbation AT1 -> AT2.png"
)

# Strunz

In [ ]:
strunz = TopObject.TopObject("Strunz", keep=True, skipProcess=True, maxSamples=None)
strunz.metadata

In [ ]:
strunz.annotations.value_counts()

In [ ]:
includeCriteria = strunz.annotations.isin(["AT1", "AT2", "Krt8+ ADI", "Basal", "Ciliated", "Club", "Goblet"])
strunz.filter(condition=includeCriteria, maxSamples=None)
# strunz.filterBestGenes(0.2)
strunz.setBasis(includeCriteria=includeCriteria, threshold=150, maxSamples=1000)

In [ ]:
includeCriteria = strunz.annotations.isin(["AT1", "AT2", "Krt8+ ADI", "Ciliated", "Club", "Goblet"])
# includeCriteria = ~strunz.annotations.isin(["AT2 activated", "Mki67+ Proliferation", "Basal"]
strunz.testBasis(seed=4, maxBasisSamples=500, maxTestSamples=500, trialCount=5, includeCriteria=includeCriteria)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(strunz, title="Strunz Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Strunz Confusion Matrix Downsampled Basis 500 Test 500 Trials 5.png"
)

In [ ]:
# strunz.project(planer.basis, "Planer")
# strunz.project(planerFilt, "PlanerFilt")
# strunz.project(NP, "Negretti-Plasschaert")
strunz.project(NM, "Negretti-Montoro")
# strunz.project(NMFull, "Negretti-Montoro Full")
# strunz.project(negrettiMontoroTurnover.basis, "NMT 0.2 1000")
# strunz.project(mcCall.basis, "McCall 755")

In [ ]:
strunzSimilarityMap = SimilarityHelper.getMatchingProjections(strunz, "Negretti-Montoro",) #includeCriteria=strunz.annotations.isin(strunz.toKeep))
SimilarityHelper.similarityBoxplot(strunzSimilarityMap, title="Strunz vs Negretti-Montoro Similarity Boxplot")#, outFile="../../PendingResults/Bharat vs LungMAP + Habermann Boxplot.png")

In [ ]:
strunzSimilarityMap = SimilarityHelper.getMatchingProjections(strunz, "McCall") #includeCriteria=strunz.annotations.isin(strunz.toKeep))
SimilarityHelper.similarityBoxplot(strunzSimilarityMap, title="Strunz vs McCall Similarity Boxplot", 
                                   # outFile="../../PendingResults/Strunz vs McCall 500 Boxplot.png"
)

In [ ]:
# plt.hist(strunz.projections["PlanerFilt"].loc["Krt5", :][strunz.annotations == "Krt8+ ADI"], bins=30, density=True, stacked=True, cumulative=True)
plt.hist(strunz.metadata[strunz.timeColumn][strunz.annotations == "Krt8+ ADI"])#, bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
axis1 = 'Krt5'
axis2 = 'AT2'
ax = SimilarityHelper.plotTwo(strunz, "Planer", axis1, axis2,
                         includeCriteria=strunz.annotations.isin(strunz.toKeep),
                         # outFile="../../PendingResults/Strunz vs Planer Krt5 vs AT2.png",
                         # includeCriteria=includeCriteria, maxLabelCount=500, unsupervisedContour=True
)

In [ ]:
SimilarityHelper.plotTwoMultiple(
    strunz, "McCall 755", 'Alveolar intermediate', 'AT2',
    includeCriteria=strunz.annotations.isin(strunz.toKeep),
    outFile='../PendingResults/Strunz vs McCall 500 Alveolar intermediate vs AT2 All Days.png'
)

In [ ]:
SimilarityHelper.plotTwoMultiple(
    projections_strunz_combined,
    'Transdifferentiating Type II', 'Cell Cycle Arrest Type II',
    strunzAnnotations,
    strunz.obs["time_point"], daysSorted,
    maxSimilarity = 0.3
)

fig.suptitle("Strunz vs Riemondy Transdifferentiating Types Over Time", fontsize=36)
plt.savefig('../PendingResults/All Days Strunz vs Riemondy Transdifferentiating Type II vs Cell Cycle Arrest Type II.png')
plt.show()

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
includeCriteria = strunz.annotations.isin(strunz.toKeep)
axis1 = 'AT1'
axis2 = 'AT2'
SimilarityHelper.plotTwoMultiple(
    strunz, "Negretti-Montoro",
    axis1, axis2, axisFontSize=24,
    includeCriteria=includeCriteria,
    legendFontSize=22, #legendMarkerScale=3,
    plotInRow=False, unsupervisedContour=False
)

In [ ]:
timePoints = strunz.obs["time_point"]
includeCriteria = np.logical_and(timePoints!='day 28', timePoints!='d14_PBS')
fig, ax = SimilarityHelper.plot_proportions(strunzAnnotations[includeCriteria], timePoints[includeCriteria], daySort, rawCounts=True)
plt.title("Strunz Source Labels Over Time")
plt.savefig('../Strunz/results/Strunz Source Labels Time Raw Counts Proportions Plot', bbox_inches='tight')
plt.show()

In [ ]:
# strunzAve, strunz_dfAve, strunz_metadataAve, strunzDataAve, strunzAnnotationsAve, strunz_kwargsAve, strunzKeptAve = SimilarityHelper.process("Strunz", filteringAnnObject=True, simplifying=False, keepAll=False, useAverage=True)
timePoints = strunz.obs["time_point"]
includeCriteria = np.logical_and(timePoints!='day 28', timePoints!='d14_PBS')
projections_strunzAve = SimilarityHelper.getTimeAveragedProjections(
    cleanedBasis, strunz_df, strunz_metadata["cell_type"], 
    timePoints[includeCriteria], daySort)

In [ ]:
toAssess = "AT2 activated"
basisKeep = ["MC-KO AT1", "MC-KO AT2", "MC-KO Basal", "MC-KO Club", "MC-KO Ciliated"]

fig, ax = plt.subplots(1, 1, figsize = (16, 8))
valueCountsFrame = pd.DataFrame()
for day in daysSorted:
    valueCountsFrame[day] = projections_strunzAve[toAssess + "_" + day]
reducedFrame = valueCountsFrame[valueCountsFrame.index.isin(basisKeep)]
ax.stackplot(daysSorted, reducedFrame.to_numpy(), labels=reducedFrame.index)
ax.legend(bbox_to_anchor=(1.0, 1.0))
ax.set_xlim(daysSorted[0], daysSorted[-1])
ax.set_ylim(0, 1)
plt.title("Strunz " + toAssess +  " Average Similarity Over Time")
plt.savefig('../Strunz/results/Strunz ' + toAssess +  ' Average Similarity Over Time Proportions Plot', bbox_inches='tight')
plt.show()

In [ ]:
revisedProjections = projections_strunz.loc[:, strunz_annotations!='Other']
xLabel = 'MC-KO Basal'
yLabel = 'MC-KO AT1'
zLabel = 'MC-KO AT2'
legendTitle = "Strunz Annotations"
figureTitle = "Strunz vs MC-KO Basis Similarity Plot"
color = strunz_annotations[strunz_annotations!='Other']
SimilarityHelper.plotThree(revisedProjections, xLabel, yLabel, zLabel, color, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
# CriticalityHelper.filterDFByTime(strunzHighVariance, day, days)
# strunzHighVariance.loc[:, timePoints == "day 2"]

In [ ]:
DNB = ["Krt8", "Fgfr2", "Krt17", "Sftpc", "Nkx2-1", "Arfgef1"]
clustersKept = ["AT2", "AT2 activated", "Krt8+ ADI"]
timePoints = strunz.obs["time_point"]
strunzHighVariance = strunz_df.loc[:, strunz_metadata["cell_type"].isin(clustersKept)]
strunzHighVariance = CriticalityHelper.filterDFByGeneVariance(strunzHighVariance, threshold = 0.3)
# strunzHighVariance = CriticalityHelper.filterDFByGeneVariance(strunz_df, threshold = 0.1)
dayEntropiesMap = {}
for day in daysSorted:
    print(day)
    entropyList = []
    strunzTimeVariance = CriticalityHelper.filterDFByTime(strunzHighVariance, day, timePoints)
    for gene in strunzHighVariance.index:
        entropyList.append(CriticalityHelper.getGeneEntropy(strunzTimeVariance, gene))
    # for gene in DNB:
    #     entropyList.append(CriticalityHelper.getGeneEntropy(strunzTimeVariance, gene))
    # entropyList.append(CriticalityHelper.getGeneEntropy(strunzTimeVariance, "Trp53"))
    dayEntropiesMap[day] = entropyList

entropyFrame = pd.DataFrame(dayEntropiesMap)
entropyFrame.index = strunzHighVariance.index
entropyFrame

In [ ]:
DNB = ["Krt8", "Fgfr2", "Krt17", "Sftpc", "Nkx2-1", "Arfgef1"]
dayGeneMap = {}
for day in daysSorted:
    dayGeneMap[day] = []
    for gene in DNB:
        if gene in entropyFrame.index:
            dayGeneMap[day].append(entropyFrame.loc[gene, day])
entropyFrame.loc[entropyFrame.index.isin(DNB), :]
# entropyFrame.loc[entropyFrame.index.isin(DNB), :].to_csv('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/BibekPneumonectomy/results/TomatoExpressing2point5GeneEntropiesTopMarkers.csv', index=True)

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(strunz, strunz.basis, "Krt8+ ADI", "AT1", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "Krt8+ ADI → AT1", 
                              # outFile="../../PendingResults/Strunz Perturbation AT1 -> AT2.png"
)

In [ ]:
np.linspace(0, 0.8, 17) 

In [ ]:
strunz.setBasis(maxSamples=None)
cellTypes = ["AT1", "AT2", "Krt8+ ADI", "AT2 activated", "Ciliated"]
dimensionsMap = Perturbation.getSpaceDimensionsAll(strunz, strunz.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False, cellTypes=cellTypes)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, 
                                 # outFile="../../PendingResults/Strunz Perturbation Dimensions.png"
)

In [ ]:
includeCriteria = strunz.annotations.isin(["AT1", "AT2", "Basal", "Ciliated", "AT2 activated", "Krt8+ ADI"])
CriticalityHelper.stateDistancePlot(strunz, "Negretti-Montoro", quantile=0.5, metric="cosine", includeCriteria=includeCriteria, title="Strunz vs Negretti-Montoro Distances", 
                                    outFile="../../PendingResults/Strunz vs Negretti-Montoro Min Cosine Distance 50% Closest.png"
)

# Kostas

In [ ]:
kostas = TopObject.TopObject("KostasI73T", exclude=True)
kostas.metadata

In [ ]:
kostas.annotations.value_counts()

In [ ]:
kostas.metadata["type"].value_counts()

In [ ]:
kostas.annotations[kostas.metadata["type"] == "WT"].value_counts()

In [ ]:
includeCriteria=kostas.annotations != "Activated AT2"
kostas.testBasis(maxBasisSamples=250, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(kostas, title="Alysandratos Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/KostasI73T Confusion Matrix Downsampled Basis 250 Test 500 Trials 5.png"
)

In [ ]:
SimilarityHelper.plotBasisCorrelationMatrix(kostas, figX=15, figY=15)

In [ ]:
kostas.project(simplifiedMouseBasis, "MC-KO")
# kostas.project(riemondy.combinedBases["MC-KO"], "RiemondyCombined")

In [ ]:
# First step: Create map of basis labels to the projections of cells with each source label onto said basis labels
kostasSimilarityMap = SimilarityHelper.getMatchingProjections(kostas, kostas.projections["RiemondyCombined"])

# Second step: Set basic parameters like plot size and title and generate boxplot
SimilarityHelper.similarityBoxplot(kostasSimilarityMap, title="Kostas vs MC-KO Similarity Boxplot", outFile="../../PendingResults/KostasI73T vs MC-KO + Riemondy Boxplot.png")

In [ ]:
axis1 = "AT1"
axis2 = "AT2"
includeCriteria = kostas.metadata["type"] != "WT"
SimilarityHelper.plotTwoObj(kostas, "MC-KO", axis1, axis2,
         unsupervisedContour=True, includeCriteria=includeCriteria,
         title="Kostas Case vs MC-KO Basis Similarity Plot", outFile="../../PendingResults/KostasI73T Case vs MC-KO AT1 vs AT2 Unsupervised.png"
)

In [ ]:
SimilarityHelper.plotTwo(kostas.projections["MC-KO"], kostas.annotations, 'AT1', 'AT2',
         gene='Krt17', geneExpressions=kostas.processed)

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'
axis3 = 'Basal'

legendTitle = "Kostas Annotations"
figureTitle = "Kostas vs MC-KO 3D Similarity Plot"
SimilarityHelper.plotThree(kostas.projections["MC-KO"], axis1, axis2, axis3, kostas.annotations, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
SimilarityHelper.plotTwo(projections_kostas_combined.loc[:,kostasAnnotations!='Other'],
         'Transdifferentiating Type II', 'Cell Cycle Arrest Type II',
         ax=ax, hue=kostasAnnotations[kostasAnnotations!='Other'],
         s=40, style=kostasAnnotations[kostasAnnotations!='Other'],
         minSimilarity=-0.2
        )
plt.legend(bbox_to_anchor=(1,1))
plt.title("Kostas vs Riemondy Transdiffrentiating Types")
plt.savefig('../PendingResults/Kostas vs MC-KO Transdifferentiating Type II vs Cell Cycle Arrest Type II')
plt.show()

In [ ]:
revisedProjections = projections_kostas.loc[:, kostas_annotations!='Other']
xLabel = 'MC-KO Basal'
yLabel = 'MC-KO AT1'
zLabel = 'MC-KO AT2'
legendTitle = "Kostas Annotations"
figureTitle = "Kostas vs MC-KO Basis Similarity Plot"
color = kostas_annotations[kostas_annotations!='Other']
SimilarityHelper.plotThree(revisedProjections, xLabel, yLabel, zLabel, color, figureTitle=figureTitle, legendTitle=legendTitle)

# Bibek

In [ ]:
bibek = TopObject.TopObject("Bibek", skipProcess=True, keep=True)#, exclude=["Activated AT2", "Proliferating AT2"]) # tomatoMarker = "TDTOMATO-EXTRA-IVS"
bibek.metadata

In [ ]:
bibek.annotations.value_counts()

In [ ]:
bibekSmall = bibek.copy()
bibekSmall.filter(maxSamples=300)
bibekSmall.setBasis()

In [ ]:
%aimport AttractorNetwork
topObject = bibekSmall
attractors = [topObject.basis["AT1"], topObject.basis["AT2"], topObject.basis["Krt8 high AT2"]]#, topObject.basis["Ciliated"], # topObject.basis["Secretory"]]
# attractors = [topObject.basis["Krt8 high AT2"], topObject.basis["AT2"], topObject.basis["AT1"]]#, topObject.basis["Krt8 high AT2"], # topObject.basis["Ciliated"]]

# attractors.columns = ['AT1', 'AT2']#, 'Ciliated', 'Secretory']
initialState = topObject.basis["Krt8 high AT2"]
attractor_network = AttractorNetwork.AttractorNetwork(attractors, initialState, 0, 0, "Triple Cusp")

In [ ]:
attractor_network.corr

In [ ]:
rng = np.random.default_rng()
simulation_args = {'total_time': 1000,
                        'signal_bounds': {'start': {'f': 0, 'k': 0.15},
                                          'end':   {'f': 900, 'k': 900}}, # k = 0 or 400, where 400 has 20% chance and 0 80% at the end
                        'finish_params': {'f': rng.choice([0.1]), 'k': 0.15},
                        'recorded_genes': ['Sftpc', 'Ager'],
                        'add_noise': False,
                        'zeroKRate': 0.01
                        }
    
output = attractor_network.simulate(**simulation_args)
dropped_data = output.iloc[:, :3]

In [ ]:
rng = np.random.default_rng()
simulation_args = {'total_time': 1000,
                        'signal_bounds': {'start': {'f': 500, 'k': 100},
                                          'end':   {'f': 800, 'k': 300}}, # k = 0 or 400, where 400 has 20% chance and 0 80% at the end
                        'finish_params': {'f': rng.choice([0.3]), 'k': 0.3},
                        'recorded_genes': ['Sftpc', 'Ager'],
                        'add_noise': False,
                        'zeroKRate': 0
                        }
    
output = attractor_network.simulate(**simulation_args)
dropped_data = output.iloc[:, :3]

In [ ]:
from itertools import combinations
fig = plt.figure(figsize=(12,12))
dropped_data = output.iloc[:, :3]

kwargs = {"c": dropped_data.index,
          "cmap": sns.color_palette("flare", as_cmap=True),
          "alpha": 0.8,
          "s": 15,
          "edgecolors": 'white',
          "linewidth": 0.2
         }

for subplot_index, pair in enumerate(combinations(dropped_data.columns, 2)):
    ax = fig.add_subplot(2, 2, subplot_index+1)

    ax.scatter(dropped_data[pair[0]], dropped_data[pair[1]], **kwargs)
    
    ax.set_xlabel(pair[0])
    ax.set_ylabel(pair[1])
               
view = [15, -20]
ax = fig.add_subplot(2, 2, 4, projection='3d')

ax.scatter(dropped_data[dropped_data.columns[0]], dropped_data[dropped_data.columns[1]], dropped_data[dropped_data.columns[2]], **kwargs)
    
ax.view_init(elev=view[0], azim=view[1])

ax.set_xlabel(dropped_data.columns[0], labelpad=10)
ax.set_ylabel(dropped_data.columns[1], labelpad=10)
ax.set_zlabel(dropped_data.columns[2], labelpad=10)
               
plt.tight_layout()    
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,5))

# separated_trials = pd.concat([all_output.iloc[0:totalTime] for i in range(n_trials)], axis=1)
totalTime = simulation_args['total_time']
a1, a2, a0 = dropped_data.columns
plt.plot(range(totalTime), dropped_data[a0], label=a0)
plt.plot(range(totalTime), dropped_data[a1], label=a1)
plt.plot(range(totalTime), dropped_data[a2], label=a2)

plt.axvline(x=simulation_args['signal_bounds']['start']['k'], label='Signal k', color='k', linestyle=':')
plt.axvline(x=simulation_args['signal_bounds']['end']['k'], color='k', linestyle=':')
plt.axvspan(simulation_args['signal_bounds']['start']['k'], simulation_args['signal_bounds']['end']['k'], alpha=0.1, color='k')

plt.axvline(x=simulation_args['signal_bounds']['start']['f'], label='Signal f', color='k', linestyle='--')
plt.axvline(x=simulation_args['signal_bounds']['end']['f'], color='k', linestyle='--')
plt.axvspan(simulation_args['signal_bounds']['start']['f'], simulation_args['signal_bounds']['end']['f'], alpha=0.1, color='k')

plt.legend(bbox_to_anchor=(1,1), loc="upper left")
plt.ylim([-0.05, 1.05])

plt.xlabel('Simulation steps')
plt.ylabel('scTOP score ($m_{\mu}$)')

plt.tight_layout()
plt.show()

In [ ]:
plt.hist(bibek.projections["PlanerFilt"].loc["Krt5", :][bibek.annotations == "Krt8 high AT2"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
bibek.annotations.value_counts()

In [ ]:
bibek.filter(condition=bibek.metadata['orig.ident'] == "combined_Epithelial", exclude=["Activated AT2", "Proliferating AT2"], skipProcess=False)
# bibek.filter(exclude=["AT2", "Proliferating AT2"], skipProcess=True)

In [ ]:
bibek.process()
selector = SelectKBest(score_func=f_classif, k=int(0.1 * len(bibek.df.index)))
trainSelected = selector.fit_transform(bibek.processed.T, bibek.annotations)
del trainSelected
selectedFeatures = bibek.df.index[selector.get_support()]
bibek.setAnndata(bibek.anndata[:, bibek.df.index.isin(list(selectedFeatures))])
bibek.setBasis()

In [ ]:
genesSelectedFrame, geneProportionFrame = bibek.getBestGenes(proportions=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], trialCount=5)
geneProportionFrame

In [ ]:
plt.subplots(1, 1, figsize=(12, 12))
ax = sns.heatmap(geneProportionFrame.astype(float), annot=True, fmt=".2f", cmap='plasma', xticklabels=geneProportionFrame.columns, yticklabels=geneProportionFrame.index,
    annot_kws={"size": 12}, cbar=True)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel("Trial Number (which train/test split)", fontsize=16)
ax.set_ylabel("Proportion of Genes Used", fontsize=16)
plt.title("Bibek (All Labels) Basis Trials", fontsize=20)
plt.tight_layout()
plt.savefig("../../PendingResults/BibekBestGeneTrials.png")

In [ ]:
bibek.combineBases(simplifiedMouseBasis, firstKeep=["Krt8 high AT2"], name="MC-KO")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
SimilarityHelper.plotBasisCorrelationMatrix(bibek, title="Bibek Epithelial Basis Pearson Correlations", textSize=12)
plt.savefig("../../PendingResults/Bibek Epithelial Dot Product Pearson Correlation Matrix.png")

## Projections

In [ ]:
# bibek.project(NM, "Negretti-Montoro")
# bibek.project(NMFull, "Negretti-Montoro")
# bibek.project(planer.basis, "Planer 276")
# bibek.project(strunz.basis, "Strunz")
# bibek.project(mcCall.basis, "McCall 755")
# bibek.project(planerFilt, "PlanerFilt")

In [ ]:
# bibekOrtho = bibek.copy()
bibek.getOrthologs(None, inplace=True)
bibek.project(Natri, "Natri")

In [ ]:
PCA = CriticalityHelper.getPCA(bibek0.processed.T)
PCA

In [ ]:
stateCloseValuesMap = {}
for state in bibek0.sortedCellTypes:
    stateCloseValuesMap[state] = getQuantileData(bibek0.projections["MC-KO"], includeCriteria=bibek0.annotations == state, quantile=0.99)
    # stateCloseValuesMap[state] = CriticalityHelper.getQuantileData(bibek0.processed, includeCriteria=bibek0.annotations == state, quantile=0.5)

In [ ]:
minDist = CriticalityHelper.getDistanceMap(stateCloseValuesMap, title="Bibek vs MC-KO Distances", method="mean",
                                                  outFile="../../PendingResults/Bibek vs MC-KO Mean Euclidean Distance 50% Closest.png"
)

In [ ]:
CriticalityHelper.selfMeanDistancePlot(stateCloseValuesMap)

In [ ]:
includeCriteria = ~bibek.annotations.isin(["Activated AT2", "Proliferating AT2"])
bibek.testBasis(seed=7, maxBasisSamples=300, maxTestSamples=500, trialCount=5, includeCriteria=includeCriteria)

In [ ]:
SimilarityHelper.getTestAccuracies(bibek)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(bibek, title="Bibek Epithelial Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Bibek Basis Confusion Matrix Downsampled Basis 300 Test 500 Trials 5.png"
)

In [ ]:
# bibek.predictivity.loc["Krt8 high AT2", "TDTOMATO-EXTRA-IVS"]
bibek.predictivity.loc["AT1", :].sort_values(ascending=False).head(20)

In [ ]:
contributions = bibek.getScoreContributions(predictivityMatrix=predictFrame)#subsetCategory=bibek.annotations, subsetName="AT1")

In [ ]:
bibek.scoreContributions["AT2"].mean(axis=1).sort_values(ascending=False).head(20)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
label = "AT2"
SimilarityHelper.plotPredictivity(ax, bibek, label, title="Bibek Predictivity", showHigh=20)
# plt.savefig("../../PendingResults/Bibek Predictivity Plot " + label + ".png")
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(20,20))
axs = ax.flatten()
for i in range(4):
    SimilarityHelper.plotTwo(bibek.projections["Simple MC-KO"], bibek.annotations,
             'AT1', 'AT2', ax=axs[i], seed=i, maxLabelCount=2500, unsupervisedContour=True, plotMultiple=True)
plt.savefig("../../PendingResults/Bibek vs MC-KO AT1 vs AT2 Downsampled x4.png")
plt.show()

In [ ]:
tsukui = TopObject.TopObject("Tsukui", skipProcess=True, exclude=["Mesothelial", "Pericyte"])
# tsukui.setBasis()
# bibek.project(tsukui.basis, "Tsukui")

In [ ]:
# tsukui.anndata = sc.read_h5ad(tsukui.filePath)

In [ ]:
tsukui.annotations.value_counts()

In [ ]:
tsukui.testBasis()

In [ ]:
SimilarityHelper.getTestAccuracies(tsukui)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(tsukui, title="Confusion Matrix Tsukui Murine")
# plt.savefig("../../PendingResults/Tsukui Confusion Matrix (Selected Samples).png")

In [ ]:
# bibekReducedDF = bibekReducedDF.T
goodGenes = bibekReducedDF.loc[bibekReducedDF.var(axis=1) > 0.01, :].index

In [ ]:
# bibek[:, ["Krt8"]].X[2, 0]
bibekReduced

In [ ]:
import scanpy as sc
cellType = "annotation_update"
bibekReduced = bibek.copy()
# bibekReduced = bibekReduced[np.logical_or(bibek_metadata[cellType] == "AT1", bibek_metadata[cellType] == "AT2")]
timeDF = bibek_df.loc[:, np.logical_and(bibek_metadata["days"] == "Day_0", np.logical_or(bibek_metadata[cellType] == "AT1", bibek_metadata[cellType] == "AT2"))]
bibekReduced = bibekReduced[timeDF.columns, :]
bibekReducedDF = bibekReduced.to_df().T
highVarianceGenes = bibekReducedDF.loc[bibekReducedDF.var(axis=1) > 0.01, :].index
bibekReduced = bibekReduced[:, list(highVarianceGenes)] # Filter by variance
# sc.pp.normalize_total(bibekReduced, inplace=True)
# sc.pp.log1p(bibekReduced, copy=False)
sc.tl.rank_genes_groups(bibekReduced, cellType, method='t-test', use_raw=False, copy=False)
sc.pl.rank_genes_groups_dotplot(
    bibekReduced, groupby=cellType, standard_scale="var", n_genes=5
)

In [ ]:
bibekReduced.uns['rank_genes_groups'].keys()

In [ ]:
bibek.annotations[np.logical_and(bibek.metadata[bibek.timeColumn] == "Day_0", bibek.annotations == "Krt8 high AT2")] = "intermediate"
bibek.metadata[bibek.cellTypeColumn] = bibek.annotations

In [ ]:
target = "intermediate"
# target = "Krt8 high AT2"
# includeCriteria = np.logical_and(bibek.metadata[bibek.timeColumn] != "Day_0", bibek.annotations != "Proliferating AT2")
includeCriteria = ~bibek.metadata[bibek.timeColumn].isin(["Day_0", "Day_04"])
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(bibek.anndata, bibek.cellTypeColumn, target, individualCompare=True, includeCriteria=includeCriteria)
includeCriteria = np.logical_and(includeCriteria, bibek.annotations == target)
targetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, bibek.processed, includeCriteria=includeCriteria, missesAllowed=2)
targetDF

In [ ]:
# diffTableMap["Krt8 high AT2"].loc[np.logical_and(diffTableMap["Krt8 high AT2"]["logfoldchanges"].abs() > 2, diffTableMap["Krt8 high AT2"]["pvals_adj"] < 0.05), :]
targetDF.loc[np.logical_and(targetDF["logfoldchanges AT1"].abs() > 2, targetDF["logfoldchanges AT2"].abs() > 2), :].to_csv("../../PendingResults/High Differential Expression Combined Control Intermediate.csv")

In [ ]:
targetDF.to_csv("../../PendingResults/High Differential Expression Krt8 high AT2 (0 misses allowed).csv")
# targetDF.loc[np.logical_and(targetDF["logfoldchanges AT2"] > 2, targetDF["logfoldchanges AT1"] > 2), :].to_csv("../../PendingResults/High Overexpression Krt8 high AT2 (2 misses allowed).csv")

In [ ]:
cellType = "annotation_update"
# timeDF = bibek_df.loc[bibek_df.index.isin(genesOfInterest), np.logical_and(bibek_metadata["days"] == "Day_0", np.logical_or(bibek_metadata[cellType] == "AT1", bibek_metadata[cellType] == "AT2"))]
bibekReducedDF = bibekReduced.to_df().T
diffDF = bibekReducedDF.loc[bibekReducedDF.index.isin(reducedDiff['names'])]
# timeDF = bibek_df.loc[bibek_df.index.isin(reducedDiff['names']), np.logical_and(bibek_metadata["days"] == "Day_0", np.logical_or(bibek_metadata[cellType] == "AT1", bibek_metadata[cellType] == "AT2"))]
print("Corr")
corr = diffDF.T.corr().values
print("Done")


In [ ]:
pdist_uncondensed = 1.0 - abs(corr)
pdist_condensed = np.concatenate([row[i+1:] for i, row in enumerate(pdist_uncondensed)])
pdist_condensed

In [ ]:
pd.DataFrame(pdist_uncondensed)

In [ ]:
# import scipy.cluster.hierarchy as spc
# linkage = spc.linkage(pdist_condensed, method='complete')
# idx = spc.fcluster(linkage, 0.5 * pdist_condensed.max(), 'distance')
idx

In [ ]:
# unique_elements, counts_elements = np.unique(idx, return_counts=True)
# # To display as a combined array (optional)
# combined_array = np.asarray((unique_elements, counts_elements)).T
# print("\nCombined array (value, count):\n", combined_array)
clusterDict = {}
for i in range(len(idx)):
    cluster = int(idx[i])
    gene = reducedDiff['names'].iloc[i]
    if cluster not in clusterDict.keys():
        clusterDict[cluster] = [gene]
    else:
        clusterDict[cluster].append(gene)
        
clusterDict

In [ ]:
output = CriticalityHelper.findDNB(bibek, "days", bibekDaysSorted, "annotation_update", "AT1", "AT2")

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
# includeCriteria =  ~bibek_metadata["annotation_update"].isin(epithelial)
cluster1 = 'Fibrotic'
cluster2 = 'Adventitial'
SimilarityHelper.plotTwoMultiple(
    projections_bibek_tsukui.loc[:, includeCriteria],
    cluster1, cluster2,
    bibekAnnotations[includeCriteria],
    bibekDays[includeCriteria], bibekDaysSorted,
    legendFontSize=12,
    minSimilarity=-0.3, maxSimilarity=0.8
)

fig.suptitle("Bibek vs Tsukui Basis " + cluster1 + " vs " + cluster2, fontsize=32)
# plt.savefig('../BibekPneumonectomy/results/All Days Bibek vs Tsukui ' + cluster1 + ' vs ' + cluster2 + '.png')
plt.show()

In [ ]:
epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
includeCriteria =  ~bibek.annotations.isin(epithelial)
basisKept = list(set(tsukui.annotations))
cell_type_column = "annotation_update"
bibekKept = [celltype for celltype in list(set(bibek.annotations)) if celltype not in epithelial]
# projections = projections_bibek_tsukui.loc[:, includeCriteria]
fig, ax = plt.subplots(figsize=(30, 12))
bibekSimilarityMap = SimilarityHelper.getMatchingProjections(bibek, projections=bibek.projections["Tsukui"], basisKeep=basisKept, sourceKeep=bibekKept)
SimilarityHelper.similarityBoxplot(fig, ax, bibekSimilarityMap, showOutliers=False)
plt.savefig('../PendingResults/Bibek vs Tsukui Similarity Boxplot (No Outliers).png')
plt.show()

In [ ]:
# day = "Day_0"
gene = "Krt8"
gene2 = "Scgb3a2"
cluster = "AT2"
# timeGeneClusterSpecificBibek = bibek_df.loc[gene, np.logical_and(bibekDays == day, bibekAnnotations == cluster)]
timeClusterSpecificBibek = CriticalityHelper.filterDFByTimeAndCluster(bibek_df, bibekDays, day, bibekAnnotations, cluster)
# timeGeneClusterSpecificBibek = CriticalityHelper.filterDFByGene(timeClusterSpecificBibek, gene)
# timeGene2ClusterSpecificBibek = CriticalityHelper.filterDFByGene(timeClusterSpecificBibek, gene2)
# timeGeneClusterSpecificBibek.corr(timeGene2ClusterSpecificBibek)
# np.corrcoef(timeGene2ClusterSpecificBibek, timeGeneClusterSpecificBibek)

timeClusterSpecificBibek.loc[timeClusterSpecificBibek.var(axis=1) > 0.3, :]

In [ ]:
corrMeans = []
cluster = "Krt8 high AT2"
# gene = "TDTOMATO-EXTRA-IVS"
gene = "Krt8"

geneExpressionCluster = CriticalityHelper.filterDFByGeneExpression(bibek_df, gene, threshold=1.5)

for day in bibekDaysSorted:
    timeClusterSpecificBibek = CriticalityHelper.filterDFByTime(geneExpressionCluster, day, bibekDays)
    # timeClusterSpecificBibek = CriticalityHelper.filterDFByTimeAndCluster(bibek_df, day, bibekDays, cluster, bibekAnnotations)
    timeClusterBibekHighVariance = CriticalityHelper.filterDFByGeneVariance(timeClusterSpecificBibek, threshold = 0.4)
    correlationList, corrMean = CriticalityHelper.getInternalCorrelationManyToMany(timeClusterBibekHighVariance)
    corrMeans.append(corrMean)
corrMeans

In [ ]:
# bibekHighVariance = CriticalityHelper.filterDFByGeneVariance(bibek_df, threshold = 0.2)
bibekExpressionVariance = CriticalityHelper.filterDFByGeneExpression(bibek_df, "TDTOMATO-EXTRA-IVS", threshold=2.5)
# bibekExpressionVariance = CriticalityHelper.filterDFByGeneExpression(bibekHighVariance, "TDTOMATO-EXTRA-IVS", threshold=2.5)
# bibekClusterExpressionVariance = bibekExpressionVariance.loc[:, bibek_metadata["annotation_update"] == "AT2"]
dayEntropiesMap = {}
for day in bibekDaysSorted:
    # if day != "Day_0":
    #     break
    print(day)
    entropyList = []
    # bibekTimeExpressionVariance = CriticalityHelper.filterDFByTime(bibekClusterExpressionVariance, day, bibekDays)
    bibekTimeExpressionVariance = CriticalityHelper.filterDFByTime(bibekExpressionVariance, day, bibekDays)
    # for gene in bibekExpressionVariance.index:
    #     entropyList.append(CriticalityHelper.getGeneEntropy(bibekTimeExpressionVariance, gene))
    entropyList.append(CriticalityHelper.getGeneEntropy(bibekTimeExpressionVariance, "Trp53"))
    dayEntropiesMap[day] = entropyList

entropyFrame = pd.DataFrame(dayEntropiesMap)
# entropyFrame.index = bibekExpressionVariance.index
entropyFrame
# day = "Day_0"
# geneExpressionCluster = CriticalityHelper.filterDFByGeneExpression(bibek_df, "TDTOMATO-EXTRA-IVS", threshold=1.5)
# timeClusterSpecificBibek = CriticalityHelper.filterDFByTime(geneExpressionCluster, day, bibekDays)
# CriticalityHelper.filterDFByGeneVariance(geneExpressionCluster, threshold = 0.3)

In [ ]:
geneList = ["FGFR2","NKX2-1","SCGB3A2","KRT8","KRT5","KRT17","SFTPC","EPCAM","TRP53","CDH1","CDH2","TDTOMATO-EXTRA-IVS","TGFB2","CDK4","LGALS1","WNT1"]
geneIDs = []

import sys
sys.path.insert(0, '../BibekPneumonectomy/scripts')
%aimport SpatialHelper
geneIDs = SpatialHelper.getMouseGeneIDs(geneList)
geneIDs

In [ ]:
# for gene in entropyFrame.index:
#     # if entropyFrame.loc[gene, "Day_0"] > 0.01:
#     if np.std(entropyFrame.loc[gene, :]) > 1.75:
#         print(gene)
entropyFrame.to_csv('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/BibekPneumonectomy/results/TomatoExpressing2point5GeneEntropies.csv', index=True)


In [ ]:
DNB = ["Krt8", "TDTOMATO-EXTRA-IVS", "Fgfr2", "Krt17", "Sftpc", "Nkx2-1", "Arfgef1"]
dayGeneMap = {}
for day in bibekDaysSorted:
    dayGeneMap[day] = []
    for gene in DNB:
        if gene in entropyFrame.index:
            dayGeneMap[day].append(entropyFrame.loc[gene, day])
entropyFrame.loc[entropyFrame.index.isin(DNB), :]
# entropyFrame.loc[entropyFrame.index.isin(DNB), :].to_csv('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/BibekPneumonectomy/results/TomatoExpressing2point5GeneEntropiesTopMarkers.csv', index=True)


In [ ]:
corrMeans = []
cluster = "Krt8 high AT2"
gene = "TDTOMATO-EXTRA-IVS"
geneOfInterest = "Krt8"
DNB = ["Krt8", "TDTOMATO-EXTRA-IVS", "Fgfr2", "Krt17", "Krt7", "Scgb3a2", "Sftpc", "Nkx2-1"]
BibekDNB = CriticalityHelper.filterDFByGenes(bibek_df, DNB)
corrList, corrMean = CriticalityHelper.getInternalCorrelationManyToMany(BibekDNB)
sum(corrList) / (len(DNB)**2)
corrMean
# corrList = CriticalityHelper.getExternalCorrelationOneToMany(bibek_df, day, bibekDays, geneOfInterest,
#                                      # clusterName=None, clusterValues=None,
#                                      clusterGene=gene, expressionThreshold=1.5,
#                                      varianceThreshold=0.5)

# # corrList, corrMean = CriticalityHelper.getExternalCorrelationManyToMany(bibek_df, day, bibekDays,
# #                                      # clusterName=None, clusterValues=None,
# #                                      gene=gene, expressionThreshold=1.5,
# #                                      varianceThreshold=0.5)
# corrList

In [ ]:
corrMeans = []
cluster = "Krt8 high AT2"
gene = "TDTOMATO-EXTRA-IVS"
geneOfInterest = "Krt8"
DNB = ["Krt8", "TDTOMATO-EXTRA-IVS", "Fgfr2", "Krt17", "Krt7", "Scgb3a2", "Sftpc", "Nkx2-1"]
corrList, externalCount = CriticalityHelper.getExternalCorrelationManyToMany(bibek_df, day, bibekDays, DNB=DNB, varianceThreshold=0.1)
externalCount

In [ ]:
DNB = ["Krt8", "TDTOMATO-EXTRA-IVS", "Fgfr2", "Sftpc", "Nkx2-1"]
day = "Day_0"
bibekExpressionVariance = CriticalityHelper.filterDFByGeneExpression(bibek_df, "TDTOMATO-EXTRA-IVS", threshold=2.5)

outputs = []
for day in bibekDaysSorted:
    outputs.append(CriticalityHelper.getSummaryValue(bibekExpressionVariance, day, bibekDays, DNB, summaryType="CI"))

for output in outputs:
    print(output[0])

In [ ]:
outputs

In [ ]:
outputDF = pd.DataFrame(outputs)
outputDF.columns = ["CI", "sd", "PCCin", "PCCout"]
outputDF.index = bibekDaysSorted
outputDF

In [ ]:
CriticalityHelper.getClusterCoefficientOfVariation(CriticalityHelper.filterDFByGenes(bibek_df, DNB))

## Main Figures

In [ ]:
# includeCriteria = bibek.metadata[bibek.timeColumn] == "Day_0"
# includeCriteria = ~bibek.metadata[bibek.timeColumn].isin(["Day_0", "Day_04"])
bibekSimilarityMap = SimilarityHelper.getMatchingProjections(bibek, "Negretti-Montoro Full", testKeep=bibek.toKeep, includeCriteria=None)
SimilarityHelper.similarityBoxplot(bibekSimilarityMap, title="Bibek vs Negretti-Montoro Similarity Boxplot", 
                                   # outFile="../../PendingResults/Bibek vs Negretti-Montoro Boxplot.png"
)

In [ ]:
# includeCriteria = bibek.metadata[bibek.timeColumn] == "Day_0"
# includeCriteria = ~bibek.metadata[bibek.timeColumn].isin(["Day_0", "Day_04"])
bibekSimilarityMap = SimilarityHelper.getMatchingProjections(bibek, "Planer 276", testKeep=bibek.toKeep, includeCriteria=None)
SimilarityHelper.similarityBoxplot(bibekSimilarityMap, title="Bibek vs Planer 276 Similarity Boxplot", 
                                   # outFile="../../PendingResults/Bibek vs Negretti-Montoro Boxplot.png"
)

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
# includeCriteria = bibek.annotations.isin(epithelial)
SimilarityHelper.plotTwoMultiple(
    bibek, "Negretti-Montoro", "AT1", "AT2", 
    unsupervisedContour=True, #gene="TDTOMATO-EXTRA-IVS",
    # includeCriteria=includeCriteria,
    # legendFontSize=22, legendMarkerScale=0.5, #title="Bibek vs MC-KO Basis Similarity Plots Over Time",
    alpha=1, markerSize=150, legendMarkerScale=0.25, axisFontSize=32, legendFontSize=32, DPI=300,
    plotInRow=True, outFile="../../PendingResults/Bibek vs NM AT1 vs AT2 All Days Unsupervised.png"
)

In [ ]:
# bibekOrtho.processed.loc[len(bibekOrtho.processed)] = bibek.processed.loc["TDTOMATO-EXTRA-IVS", :]
bibekOrtho.processed = bibekOrtho.processed.rename(index={13954:"TDTOMATO-EXTRA-IVS"})
bibekOrtho.processed

# bibek.processed.loc["TDTOMATO-EXTRA-IVS", :]

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
# includeCriteria = bibek.annotations.isin(epithelial)
includeCriteria = ~bibek.annotations.isin(["Secretory", "Proliferating AT2", "Activated AT2"])
SimilarityHelper.plotTwo(
    bibek, "Natri", 'AT1', 'AT2', 
    unsupervisedContour=False, maxLabelCount=2500, #gene="TDTOMATO-EXTRA-IVS",
    includeCriteria=includeCriteria, DPI=300,
    # legendFontSize=22, legendMarkerScale=0.5, #title="Bibek vs MC-KO Basis Similarity Plots Over Time",
    # markerSize=150, legendMarkerScale=0.25, axisFontSize=32, legendFontSize=32, DPI=300, #maxLabelCount=100
    # plotInRow=True, 
    outFile="../../PendingResults/Bibek vs Natri AT1 vs AT2.png"
)

In [ ]:
plt.hist(bibek.processed.loc["Cldn4", bibek.annotations == "AT1"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
plt.hist(bibek.processed.loc["Cldn4", bibek.annotations == "Krt8 high AT2"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
includeCriteria = ~bibek.annotations.isin(["Secretory", "Proliferating AT2", "Activated AT2"])
ax = SimilarityHelper.plotTwo(bibek, "Negretti-Montoro", "AT1", "AT2", 
                        includeCriteria=includeCriteria, DPI=300, title="Thapa PNX projected on Negretti-Montoro", 
                        outFile="../../PendingResults/Bibek vs NM AT1 vs AT2 Simple.png", legendInner=True,
)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(35,20))
marker_genes = ['Hopx', 'Tead1', 'Vegfa', 'Etv5', 'Cebpa', 'Cldn4']
axs = ax.flatten()
for i, gene in enumerate(marker_genes):
    SimilarityHelper.plotTwo(bibek, "MC-KO", 'AT1', 'AT2',
             gene=gene, ax=axs[i], show=False, 
                             includeCriteria=bibek.metadata[bibek.timeColumn] == "Day_0"
)
# plt.savefig('../PendingResults/Bibek Day 0 AT1 and AT2 Markers.png')
plt.show()

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'

SimilarityHelper.plotTwoMultiple(bibek, "MC-KO",
    axis1, axis2,
    axisFontSize=24, legendFontSize=22, legendMarkerScale=3, markerSize=80,
    gene="Hopx"
)

In [ ]:
"Ascl3" in bibek.df.index

In [ ]:
SimilarityHelper.plotTwoMultiple(
    projections_bibek_combined, 
    'Transdifferentiating Type II', 'Cell Cycle Arrest Type II', 
    bibekAnnotations,
    bibekDays, bibekDaysSorted,
    legendFontSize=12,
    minSimilarity=-0.15, maxSimilarity=0.3
)

fig.suptitle("Bibek vs MC-KO Basis Similarity Plots Over Time", fontsize=36)
plt.savefig('../BibekPneumonectomy/results/All Days Bibek vs MC-KO Transdifferentiating Type II vs Cell Cycle Arrest Type II.png')
plt.show()

In [ ]:
SimilarityHelper.plotTwoMultiple(
    projections_bibek_combined, 
    'Transdifferentiating Type II', 'Cell Cycle Arrest Type II', 
    bibekAnnotations,
    bibekDays, bibekDaysSorted,
    legendFontSize=12,
    minSimilarity=-0.15, maxSimilarity=0.3,
    gene=tomatoMarker, sourceData = bibekData
)

fig.suptitle("Bibek vs MC-KO Basis Similarity Plots Over Time", fontsize=36)
plt.savefig('../BibekPneumonectomy/results/Tomato Marker All Days Bibek vs MC-KO Transdifferentiating Type II vs Cell Cycle Arrest Type II.png')
plt.show()

In [ ]:
epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
includeCriteria = bibek.annotations.isin(epithelial)
ax = SimilarityHelper.plotTwo(bibek, "MC-KO", "AT1", "AT2",
                    includeCriteria=includeCriteria,
                    unsupervisedContour=False,
                    title="Bibek Control Projected Onto Herriges Reference",
                    outFile='../PendingResults/AT1 vs AT2 Bibek vs MC-KO.png'
)

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
# includeCriteria = bibek.annotations.isin(epithelial)
axis1 = 'AT1'
axis2 = 'AT2'
caption = "Lung epithelial scRNAseq data of mice injured via pneumonectomy, a weak injury, from Thapa et al. plotted against reference basis built from Negretti et al and Plasschaert et al. Krt8 high AT2 represents a typical intermediate in transdifferentiation between AT2 and AT1, and it fails to form a cluster at any time point."

SimilarityHelper.plotTwoMultiple(
    bibek, "NP",
    axis1, axis2, axisFontSize=40, titleFontSize=44,
    # includeCriteria=includeCriteria,
    legendFontSize=40, legendMarkerScale=0.5, markerSize=80, title="Thapa vs Negretti-Plasschaert Murine Basis Similarity Plot", #caption=caption,
    xBounds=(-0.5, 0.9), yBounds=(-0.5, 0.9), plotInRow=False, unsupervisedContour=True, seed=1, #maxLabelCount=600
    # outFile='../PendingResults/Bibek vs NP AT1 vs AT2 All Days Unsupervised Contour (Downsampled 600).png'
)

In [ ]:
# bibek.project(riemondy.combinedBases["MC-KO"], "MC-KO + Cell Cycle Arrest")
bibek.project(riemondy.combinedBases["MC-KO T"], "MC-KO T")

In [ ]:
# epithelial = ["AT1", "AT2", "Krt8 high AT2", "Activated AT2", "Ciliated", "Proliferating AT2", "Secretory"]
# includeCriteria = bibek.annotations.isin(epithelial)
axis1 = 'Transdifferentiating Type II'
axis2 = 'AT2'
SimilarityHelper.plotTwoMultiple(
    bibek, bibek.projections["MC-KO T"],
    axis1, axis2, axisFontSize=24,
    unsupervisedContour=True,
    legendFontSize=22, legendMarkerScale=0.5, title="Bibek vs Riemondy Transdifferentiating Type II + MC-KO Basis",
    xBounds=(-0.15, 0.7), yBounds=(-0.2, 0.65), plotInRow=True
)

plt.savefig('../PendingResults/Bibek Riemondy Transdifferentiating Type II vs MC-KO AT2 All Days Unsupervised Contour.png')
plt.show()

In [ ]:
bibek.setBasis()
dimensionsMap = Perturbation.getSpaceDimensionsAll(bibek, bibek.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, outFile="../../PendingResults/Bibek Perturbation Dimensions.png")

# Kobayashi
### Notes: AT2 consistently cannot recapitulate itself

In [ ]:
kobayashi = TopObject.TopObject("Kobayashi", skipProcess=False, keep=True)
kobayashi.metadata

In [ ]:
kobayashi.annotations.value_counts()

In [ ]:
# kobayashi.project(planer.basis, "Planer")
kobayashi.project(NM, "Negretti-Montoro")

In [ ]:
includeCriteria=kobayashi.annotations.isin(["AEC2", "AEC1", "Lgals3+", "Ctgf+"])
kobayashi.testBasis(maxBasisSamples=100, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=9)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(kobayashi, title="Kobayashi Basis Confusion Matrix", decimalMode="Clean",
                                              # outFile="../../PendingResults/Kobayashi Confusion Matrix Downsampled Basis 100 Test 500 Trials 5.png"
)

In [ ]:
includeCriteria = None
# kobayashiSimilarityMap = SimilarityHelper.getMatchingProjections(kobayashi, "Negretti-Montoro", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(kobayashiSimilarityMap, 
        title="Kobayashi vs Negretti-Montoro Similarity Boxplot", testKeep=["AEC1", "AEC2", "AEC2-proliferating", "Ctgf+", "Lgals3+"]
        # outFile="../../PendingResults/Kobayashi vs NM Boxplot.png"
)

In [ ]:
plt.hist(kobayashi.projections["PlanerFilt"].loc["Krt5", :][kobayashi.annotations == "Ctgf+"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
axis1 = "AT1"
axis2 = "AT2"
ax = SimilarityHelper.plotTwo(kobayashi, "Negretti-Montoro", axis1, axis2)

In [ ]:
axis1 = "AT1"
axis2 = "AT2"
ax = SimilarityHelper.plotTwo(kobayashi, "PlanerFilt", axis1, axis2,
                              gene="Krt19",
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
SimilarityHelper.plotTwo(projections_kobayashi_combined.loc[:, kobayashiAnnotations!='Other'],
         'MC-KO AT1', 'MC-KO AT2',
         ax=ax, hue=kobayashiAnnotations[kobayashiAnnotations!='Other'],
         s=40, style=kobayashiAnnotations[kobayashiAnnotations!='Other'])
plt.legend(bbox_to_anchor=(1,1))
plt.title('Koboyashi vs MC-KO Basis Similarity Plot.png')
plt.savefig('Koboyashi vs MC-KO Basis AT1 vs AT2.png')
plt.show()

# Choi

In [ ]:
choi = TopObject.TopObject("Choi", skipProcess=True, keep=True)
choi.metadata

In [ ]:
choi.annotations.value_counts()

In [ ]:
plt.hist(choi.processed.loc["Gpx2", choi.annotations == "Intermediate"], bins=30, density=True, stacked=False, cumulative=False)
# plt.hist(choi.projections["NP"].loc["AT1", choi.annotations == "AT1"], range=(0.35, 0.8), bins=20, density=True, stacked=True, cumulative=True)

In [ ]:
# choi.setBasis()
choi.combineBases(riemondy.basis, firstKeep=['AT2', 'AT1'], secondKeep=["Basal"], name="Riemondy")

In [ ]:
choi.testBasis(seed=2, maxBasisSamples=180, maxTestSamples=500, trialCount=5, includeCriteria=choi.annotations.isin(["AT1", "AT2", "Intermediate"]))

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(choi, title="Choi Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Choi Confusion Matrix Downsampled Basis 180 Test 500 Trials 5.png"
)

In [ ]:
choi.project(NM, "Negretti-Montoro")
# choi.project(NMFull, "Negretti-Montoro")
# choi.project(planer.basis, "Planer")
# choi.project(planerFilt, "PlanerFilt")
# choiOrtho = choi.copy()
# choiOrtho.getOrthologs(mapping, inplace=True)
# choiOrtho.project(HaberMAP500, "HaberMAP500")
# choiOrtho.project(HaberMAP, "HaberMAP")

In [ ]:
choi.metadata.columns

In [ ]:
# includeCriteria = choi.metadata[choi.timeColumn] == "T_d0"
choiSimilarityMap = SimilarityHelper.getMatchingProjections(choi, "Negretti-Montoro", includeCriteria=None)
SimilarityHelper.similarityBoxplot(choiSimilarityMap, 
                                   # title="Choi vs Planer Similarity Boxplot", 
                                   # outFile="../../PendingResults/Choi Control vs Planer Boxplot.png"
)

In [ ]:
# includeCriteria = choi.metadata[choi.timeColumn] == "T_d0"
ax = SimilarityHelper.plotTwo(choi, "Negretti-Montoro", 'AT1', 'AT2',
                includeCriteria=None, unsupervisedContour=True, maxLabelCount=300, 
                # outFile='../PendingResults/Choi vs NM AT1 vs AT2 Unsupervised 300.png'
)

In [ ]:
# includeCriteria = choi.metadata[choi.timeColumn] == "T_d0"
ax = SimilarityHelper.plotTwo(choi, "Negretti-Montoro", 'AT1', 'AT2',
                includeCriteria=None, unsupervisedContour=False, maxLabelCount=None, gene="Il1r1",
                # outFile='../PendingResults/Choi vs NM AT1 vs AT2 Unsupervised 300.png'
)

In [ ]:
# includeCriteria = choi.metadata[choi.timeColumn] == "T_d0"
ax = SimilarityHelper.plotTwo(choiOrtho, "HaberMAP", 'KRT5-/KRT17+', 'AT2',
                includeCriteria=None, unsupervisedContour=True, maxLabelCount=300, 
                outFile='../PendingResults/Choi vs HaberMAP SKAR vs AT2 Unsupervised 300.png'
)

In [ ]:
SimilarityHelper.plotTwoMultiple(choi, "Negretti-Montoro", 'Basal', 'AT2', 
            xBounds=(-0.2, 0.4), yBounds=(-0.2, 0.9),
            # outFile="../../PendingResults/Choi vs NM Full Basal vs AT2.png"
)

In [ ]:
[val for val in choi.df.index if val.upper().startswith("SCGB")]

In [ ]:
SimilarityHelper.plotTwoMultiple(choi, "Negretti-Montoro", 'AT1', 'AT2', 
            xBounds=(-0.2, 0.4), yBounds=(-0.2, 0.9),
            gene="Krt15",
            # outFile="../../PendingResults/Choi vs NM Full Basal vs AT2.png"
)

In [ ]:
SimilarityHelper.plotTwoMultiple(
    projections_choi_combined, 
    'Transdifferentiating Type II', 'Cell Cycle Arrest Type II', 
    choi_metadata[cell_type_column_choi],
    choiDays, choiDaysSorted,
    legendFontSize=12,
    maxSimilarity=0.5
)

fig.suptitle("Choi vs Riemondy Transdifferentiating Types Over Time", fontsize=36)
plt.savefig('../PendingResults/All Days Choi vs Riemondy Transdifferentiating Types Over Time.png')
plt.show()

In [ ]:
SimilarityHelper.geneViolinPlot(choi, "Il1r1")

In [ ]:
DNB = ["Krt8", "Fgfr2", "Krt17", "Sftpc", "Nkx2-1", "Arfgef1"]
timePoints = choi.obs["treat"]
choiHighVariance = CriticalityHelper.filterDFByGeneVariance(choi_df, threshold = 0.3)
# choiHighVariance = CriticalityHelper.filterDFByGeneVariance(choi_df, threshold = 0.1)
dayEntropiesMap = {}
for day in choiDaysSorted:
    print(day)
    entropyList = []
    choiTimeVariance = CriticalityHelper.filterDFByTime(choiHighVariance, day, timePoints)
    for gene in choiHighVariance.index:
        entropyList.append(CriticalityHelper.getGeneEntropy(choiTimeVariance, gene))
    # for gene in DNB:
    #     entropyList.append(CriticalityHelper.getGeneEntropy(choiTimeVariance, gene))
    # entropyList.append(CriticalityHelper.getGeneEntropy(choiTimeVariance, "Trp53"))
    dayEntropiesMap[day] = entropyList

entropyFrame = pd.DataFrame(dayEntropiesMap)
entropyFrame.index = choiHighVariance.index
entropyFrame

In [ ]:
DNB = ["Krt8", "Fgfr2", "Krt17", "Sftpc", "Nkx2-1", "Arfgef1"]
dayGeneMap = {}
for day in choiDaysSorted:
    dayGeneMap[day] = []
    for gene in DNB:
        if gene in entropyFrame.index:
            dayGeneMap[day].append(entropyFrame.loc[gene, day])
entropyFrame.loc[entropyFrame.index.isin(DNB), :]

In [ ]:
revisedProjections = projections_choi.loc[:, choi_annotations!='Other']
xLabel = 'MC-KO Basal'
yLabel = 'MC-KO AT1'
zLabel = 'MC-KO AT2'
legendTitle = "Choi Annotations"
figureTitle = "Choi vs MC-KO Basis 3D Similarity Plot"
x = revisedProjections.loc[xLabel]
y = revisedProjections.loc[yLabel]
z = revisedProjections.loc[zLabel]
color = choi_annotations[choi_annotations!='Other']
SimilarityHelper.plotThree(revisedProjections, xLabel, yLabel, zLabel, color, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(8,8))
SimilarityHelper.plotTwo(choi, "MC-KO", "AT1", "AT2", annotations=choi.annotations, unsupervisedContour=True, title="Choi vs Simplified MC-KO")
# plt.savefig('../PendingResults/AT1 vs AT2 Unsupervised Contour Plot Choi vs Simplified MC-KO.png')
plt.show()

In [ ]:
includeCriteria = choi.annotations.isin(choi.toKeep)
axis1 = 'AT1'
axis2 = 'AT2'
SimilarityHelper.plotTwoMultiple(
    choi, "MC-KO",
    axis1, axis2, axisFontSize=24,
    includeCriteria=includeCriteria,
    legendFontSize=22, legendMarkerScale=2, title="Choi vs MC-KO Basis",
    similarityBounds=(-0.1, 0.4), plotInRow=True, unsupervisedContour=True
)

plt.savefig('../PendingResults/Choi vs MC-KO AT1 vs AT2 All Days Unsupervised Contour.png')
plt.show()

In [ ]:
choi.metadata

In [ ]:
choi.annotations[includeCriteria].value_counts()

In [ ]:
# includeCriteria = choi.metadata[choi.timeColumn] == "T_d0"
CriticalityHelper.stateDistancePlot(choi, "MC-KO", includeCriteria=None, quantile=0.5, title="Choi vs MC-KO Distances", outFile="Choi vs MC-KO Min Euclidean Distance 50% Closest.png")

In [ ]:
# choi.setBasis()
# dimensionsMap = Perturbation.getSpaceDimensionsAll(choi, choi.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False, cellTypes=["AT1", "AT2", "Intermediate"])
SimilarityHelper.plotSpaceMatrix(dimensionsMap, title="Choi Perturbations", outFile="../../PendingResults/Choi Perturbation Dimensions.png")

# Planer

In [ ]:
planer = TopObject.TopObject("Planer", skipProcess=True)
planer.metadata

In [ ]:
planer.annotations.value_counts()
# planer.annotations[planer.metadata[planer.timeColumn] == "0"].value_counts()

In [ ]:
planer.metadata[np.logical_and(planer.metadata[planer.timeColumn] != "0", planer.annotations == "Alveolar_transitional")]["trace_call"].value_counts()

In [ ]:
planer.metadata.columns

In [ ]:
includeCriteria = ~planer.metadata[planer.timeColumn].isin(["0"])
# plt.hist(planer.processed.loc["Tgfb1", planer.annotations == "AT1_AT2"], bins=30, density=True, stacked=True, cumulative=True, range=(0, 0.03))
# plt.hist(planer.processed.loc["Krt15", planer.annotations == "AT1_AT2"], bins=30, density=True, stacked=False, cumulative=False, range=(0, 0.06))
plt.hist(planer.processed.loc["Ndrg1", np.logical_and(includeCriteria, planer.annotations == "AT1_AT2")], bins=30, density=True, stacked=False, cumulative=False)

In [ ]:
includeCriteria = ~planer.metadata[planer.timeColumn].isin(["0"])
# plt.hist(planer.processed.loc["Krt15", planer.annotations == "Alveolar_transitional"], bins=30, density=True, stacked=False, cumulative=False, range=(0, 0.06))
plt.hist(planer.projections["Negretti-Montoro"].loc["Basal", np.logical_and(includeCriteria, planer.annotations == "Alveolar_transitional")], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
# planer = planerOG.copy()
# conditionList = [np.logical_or(~planer.annotations.isin(["AT1", "AT2"]), planer.metadata[planer.timeColumn] == "0")]
# condition = np.logical_and(condition, np.logical_or(planer.annotations != "Ciliated", planer.metadata[planer.timeColumn].isin(["0", "6"])))
# conditionList.append(np.logical_or(planer.annotations != "Ciliated", planer.metadata[planer.timeColumn].isin(["0", "6"])))
planer.filter(exclude=["AT1_AT2"], maxSamples=None, conditionList=None, skipProcess=True)
# # planer.filter(exclude=["AT1_AT2", "Alveolar_transitional"])

planerOG = planer.copy()
planer.filterBestGenes(0.1)
planer.setBasis()

In [ ]:
# planer.setBasis(allowedGenes=None)
planer.combineBases(NM, firstExclude=["Krt5", "Secretory"], secondKeep=["Basal", "Secretory"], name="NMSmall")

In [ ]:
# planerOG = planer.copy()
planer.filter(maxSamples=1000)
planer.filterBestGenes(0.2)

In [ ]:
planer.testBasis(seed=2, maxBasisSamples=220, maxTestSamples=500, trialCount=5, includeCriteria=planer.annotations != "Secretory")

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(planer, title="Niethamer Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Planer Confusion Matrix Downsampled Basis 220 Test 500 Trials 5.png"
)

## Projections

In [ ]:
# planer.project(simplifiedMouseBasis, "MC-KO")
# negrettiPlasschaert.filter(exclude=["Secretory"])
# negrettiPlasschaert.filter(maxSamples=500)
# negrettiPlasschaert.setBasis()
# planer.project(NM, "Negretti-Montoro")
# planer.project(NMFull, "Negretti-Montoro Full")
# planer.project(mcCall.combinedBases["NMMB"], "Negretti-Montoro-McCall")
# planer.project(riemondy.combinedBases["MC-KO"], "RiemondyCombined")
# planer.project(bibek.combinedBases["MC-KO"], "BibekCombined")
# planer.project(mcCall.basis, "McCall")
# planerOrtho = planer.copy()
# planerOrtho.getOrthologs(mapping, inplace=True)
# planerOrtho.project(HaberMAP500, "HaberMAP500")
planerOrtho.project(HaberMAP, "HaberMAP")

In [ ]:
# includeCriteria = planer.metadata[planer.timeColumn] == "0"
planerSimilarityMap = SimilarityHelper.getMatchingProjections(planer, "Negretti-Montoro", includeCriteria=None)
SimilarityHelper.similarityBoxplot(planerSimilarityMap, title="Planer All Days vs Negretti + Montoro Similarity Boxplot", 
                                   # outFile="../../PendingResults/Planer All Days vs NM Boxplot.png"
)

In [ ]:
# includeCriteria = planer.metadata[planer.timeColumn] == "0"
planerSimilarityMap = SimilarityHelper.getMatchingProjections(planer, "McCall", includeCriteria=None)
SimilarityHelper.similarityBoxplot(planerSimilarityMap, title="Planer vs McCall Similarity Boxplot", 
                                   outFile="../../PendingResults/Planer vs McCall Boxplot.png"
)

In [ ]:
newVals = []
for val in planer.metadata["trace_call"]:
    match val:
        case "Traced":
            newVals.append(2)
        case "Untraced":
            newVals.append(1)
        case "Not_detected":
            newVals.append(0)
# newVals
planer.metadata["trace_call_int"] = newVals

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'
includeCriteria = ~planer.annotations.isin(["Ciliated"])
ax = SimilarityHelper.plotTwo(planer, "Negretti-Montoro", axis1, axis2,
    includeCriteria=None,
    unsupervisedContour=True, maxLabelCount=300,
    title="Planer Projected Onto Negretti-Montoro Reference",
    outFile="../../PendingResults/Planer vs NM AT1 vs AT2 Unsupervised 300.png"
)

In [ ]:
axis1 = 'KRT5-/KRT17+'
axis2 = 'AT2'
includeCriteria = ~planer.annotations.isin(["Ciliated"])
ax = SimilarityHelper.plotTwo(planerOrtho, "HaberMAP", axis1, axis2,
    includeCriteria=None,
    unsupervisedContour=True, maxLabelCount=300,
    title="Planer Projected Onto HaberMAP Reference",
    outFile="../../PendingResults/Planer vs HaberMAP SKAR vs AT2 Unsupervised 300.png"
)

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'
includeCriteria = ~planer.annotations.isin(["Ciliated", "Secretory", "Krt5"])
SimilarityHelper.plotTwoMultiple(planer, "Negretti-Montoro", axis1, axis2,
    xBounds=(-0.35, 0.7), yBounds=(-0.35, 0.7), alternateColumn="trace_call_int", #gene="Plau"
    plotInRow=False, includeCriteria=includeCriteria, #unsupervisedContour=True, maxLabelCount=500,
    title="Planer vs Negretti + Plasschaert Basis", outFile="../../PendingResults/Planer vs NM AT1 vs AT2.png"
)

In [ ]:
axis1 = 'Alveolar intermediate'
axis2 = 'AT2'
includeCriteria = ~planer.annotations.isin(["Ciliated"])
SimilarityHelper.plotTwoMultiple(planer, "McCall", axis1, axis2,
    xBounds=(-0.2, 0.4), yBounds=(-0.2, 0.6), includeCriteria=None, markerSize=70,
    plotInRow=False, #unsupervisedContour=True,# maxLabelCount=500,
    title="Planer Projected Onto McCall Reference",
    outFile="../../PendingResults/Planer vs McCall Alveolar intermediate vs AT2.png"
)

In [ ]:
includeCriteria = planer.metadata[planer.timeColumn] == "0"
planer.annotations[includeCriteria].value_counts()

In [ ]:
includeCriteria = planer.metadata[planer.timeColumn] == "0"
CriticalityHelper.stateDistancePlot(planer, "MC-KO", quantile=0.8, includeCriteria=includeCriteria, 
                                    title="Planer vs MC-KO Distances", outFile="../../PendingResults/Planer vs MC-KO Min Euclidean Distance 80% Closest.png")

In [ ]:
target = "AT1_AT2"
includeCriteria = ~planer.metadata[planer.timeColumn].isin(["-1"])
diffTableMap1 = CriticalityHelper.getDifferentiallyExpressedGenes(planer.anndata, planer.cellTypeColumn, target, individualCompare=True, includeCriteria=includeCriteria)
includeCriteria = np.logical_and(includeCriteria, planer.annotations == target)
targetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap1, planer.processed, includeCriteria=includeCriteria, missesAllowed=0)
targetDF

In [ ]:
target = "Alveolar_transitional"
includeCriteria = ~planer.metadata[planer.timeColumn].isin(["0"])
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(planer.anndata, planer.cellTypeColumn, target, individualCompare=True, includeCriteria=includeCriteria)
includeCriteria = np.logical_and(includeCriteria, planer.annotations == target)
targetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, planer.processed, includeCriteria=includeCriteria, missesAllowed=1, expressionThreshold=0.01)
targetDF

In [ ]:
planer.setBasis()
# res = Perturbation.runProcessedSpaceExperiment(planer, planer.basis, "AT2", "AT1", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
# Perturbation.plot_flip_curves(res, "AT2 → AT1", 
#                               # output_file='SCGB3A2_to_AT1.png'
# )

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(planer, planer.basis, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
# planer.setBasis(includeCriteria=planer.annotations.isin(["AT2", "AT1", "AT1_AT2", "Alveolar_transitional", "Ciliated"]))
# dimensionsMap = Perturbation.getSpaceDimensionsAll(planer, planer.basis, maxCells=1000, randomState=9, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, title="Niethamer Perturbations",
                                 outFile="../../PendingResults/Planer Perturbation Dimensions.png"
)

# Auyeung

In [ ]:
auyeung = TopObject.TopObject("Auyeung", skipProcess=True)
auyeung.metadata

In [ ]:
auyeung.metadata["Seq"].value_counts()

In [ ]:
auyeung.annotations.value_counts()

In [ ]:
auyeung.metadata.columns
# auyeung.setAnndata(auyeung.anndata[:, auyeung.df.index.isin(simplifiedMouseBasis.index)])
# auyeung.setMetadata(cellTypeColumn="named_clusters_withsubclusters")

In [ ]:
auyeung.testBasis(seed=20, maxBasisSamples=350, maxTestSamples=500, trialCount=5, includeCriteria=auyeung.annotations.isin(["AT2", "Fibrotic_transitional", "Regenerative_transitional"]))
# auyeung.testBasis(seed=2, maxBasisSamples=350, maxTestSamples=500, trialCount=5, includeCriteria=auyeung.annotations.isin(["Activated_AT2", "Fibrotic_transitional", "Regenerative_transitional"]))

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(auyeung, title="Auyeung Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Auyeung Confusion Matrix Downsampled Basis 350 Test 500 Trials 5.png"
)

In [ ]:
# auyeung.process(normalize=False)
# auyeung.project(NM, "Negretti-Montoro")
# auyeung.getOrthologs(mapping, inplace=True)
# auyeung.project(HaberMAP500, "HaberMAP500")
# auyeung.project(HaberMAP, "HaberMAP")
auyeung.project(Adams, "Adams")
# auyeung.project(planer.basis, "Planer", alignGenes=False)
# auyeung.project(planerFilt, "PlanerFilt", alignGenes=False)

In [ ]:
includeCriteria = None
auyeungSimilarityMap = SimilarityHelper.getMatchingProjections(auyeung, "Negretti-Montoro", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(auyeungSimilarityMap, 
        # title="Auyeung vs Negretti-Montoro Similarity Boxplot",
        # outFile="../../PendingResults/Auyeung vs MC-KO (Updated, Smallest) Boxplot.png"
)

In [ ]:
includeCriteria = None
# auyeungSimilarityMap = SimilarityHelper.getMatchingProjections(auyeung, "McCall", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(auyeungSimilarityMap, 
        title="Auyeung vs McCall Similarity Boxplot",
        outFile="../../PendingResults/Auyeung vs McCall Boxplot.png"
)

In [ ]:
plt.hist(auyeung.projections["PlanerFilt"].loc["Krt5", :][auyeung.annotations == "Fibrotic_transitional"], bins=30, density=True, stacked=True, cumulative=True)

In [ ]:
# includeCriteria = np.logical_and(kaminski2020.annotations != "Ciliated", kaminski2020.metadata['Disease_Identity'] == "Control")
axis1 = 'AT1'
axis2 = 'AT2'

ax = SimilarityHelper.plotTwo(Auyeung, "NP", axis1, axis2,
                         title="Auyeung vs Negretti + Plasschaert Basis AT1 vs AT2", 
                         outFile="../../PendingResults/Auyeung All vs NP AT1 vs AT2.png",
                         # maxLabelCount=500,
                         # unsupervisedContour=True
)

In [ ]:
## Overlay within same dataset
topObjects = [Auyeung]
diseaseCategory = "Group"
names = ["BleoKO", "BleoKIRA8", "BleoCtrl", "CtrlCtrl"]
additionalCriteriaOthers = ~topObjects[0].annotations.isin(["Cycling", "Activated_AT2", "AT2"]) #"ATII", "ATI"
# additionalCriteriaAll = ~topObjects[0].annotations.isin(["Ciliated", "Club", "Goblet"])
additionalCriteriaAll = None
includeCriteriaList = SimilarityHelper.setupIncludeCriteria(topObjects[0], diseaseCategory, names, additionalCriteriaOthers=additionalCriteriaOthers, additionalCriteriaAll=additionalCriteriaAll)
otherProjections, otherAnnotations = SimilarityHelper.setupOverlay(topObjects, "NP", includeCriteriaList)

In [ ]:
ax = SimilarityHelper.plotTwo(Auyeung, "NP", "Basal", "AT2",
             additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
             includeCriteria=includeCriteriaList[0],
             title="Auyeung projected onto Negretti + Plasschaert Reference",
             outFile="../../PendingResults/Auyeung Bleo Overlay vs NP Basal vs AT2.png",
             # unsupervisedContour=False, maxLabelCount=500,
)

In [ ]:
ax = SimilarityHelper.plotTwo(auyeung, "HaberMAP", "KRT5-/KRT17+", "AT2",
             includeCriteria=None,
             title="Auyeung projected onto HaberMAP Reference",
             # unsupervisedContour=False, maxLabelCount=500,
             # outFile="../../PendingResults/Auyeung vs HaberMAP500 ANOVA2 SKAR vs AT2.png",
)

In [ ]:
SimilarityHelper.getProjectionStats(auyeung, "HaberMAP", "Regenerative_transitional", target="KRT5-/KRT17+",
                                    outFile="../../PendingResults/Auyeung vs HaberMAP Regenerative_transitional Stats.csv",
)

In [ ]:
auyeung.setBasis()
auyeungDimensionsMap = Perturbation.getSpaceDimensionsAll(auyeung, auyeung.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False, cellTypes=["AT2", "Fibrotic_transitional", "Regenerative_transitional"])
SimilarityHelper.plotSpaceMatrix(auyeungDimensionsMap, outFile="../../PendingResults/Auyeung Perturbation Dimensions.png")

# Zacharias

In [ ]:
# zacharias = TopObject.TopObject("Zacharias", skipProcess=True, keep=["ABI1", "ABI2", "Basal", "PATS_like"])
zacharias = TopObject.TopObject("Zacharias", skipProcess=False, keep=False, maxSamples=10000)
# zacharias.filter(exclude=["Basal"], skipProcess=True)
zacharias.metadata

In [ ]:
zacharias.annotations.value_counts()

In [ ]:
# Reannotate with new labels
newAnnotations = [val if not val in ["Krt8", "AEP"] else "AT2" for val in zacharias.annotations]
newAnnotations = [val if val != "PATS_like" else "KO_AT2" for val in newAnnotations]
newAnnotations = [val if val != "ABI1" else "ABI" for val in newAnnotations]
newAnnotations = [val if not val in ["Basal", "ABI2"] else "Basal-like" for val in newAnnotations]
zacharias.anndata.obs["celltypeNew"] = newAnnotations
zacharias.cellTypeColumn = "celltypeNew"
zacharias.setMetadata()
zacharias.annotations.value_counts()

In [ ]:
# zacharias.filter(maxSamples=1000, skipProcess=True)
# zacharias.filterBestGenes(0.1)
# zacharias.setBasis()
zacharias.setAnndata(zacharias.anndata[zacharias.annotations != "AT1"])
# zacharias.combineBases(NMFull, firstKeep=["ABI"], name="ZNM")

In [ ]:
includeCriteria=zacharias.annotations != "KO_AT2"
zacharias.testBasis(maxBasisSamples=200, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(zacharias, title="Zacharias Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/Zacharias Confusion Matrix Downsampled Basis 200 Test 500 Trials 5.png"
)

In [ ]:
plt.hist(zacharias.processed.loc["Sfn", zacharias.annotations == "AT2"], bins=30, density=True, stacked=False, cumulative=False, range=(0, 0.03))

In [ ]:
plt.hist(zacharias.processed.loc["Sfn", zacharias.annotations == "ABI1"], bins=30, density=True, stacked=False, cumulative=False, range=(0, 0.03))

In [ ]:
# plt.hist(zacharias.metadata["nFeature_RNA"])
[val for val in zacharias.df.index if val.upper().startswith("TD")]

In [ ]:
target = "ABI1"
# includeCriteria = zacharias.annotations.isin(["Ciliated", "Secretory", target])
includeCriteria = None
zachariasDiffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(zacharias.anndata, zacharias.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria
)

# includeCriteria = np.logical_and(includeCriteria, plasschaert.annotations == target)
includeCriteria = zacharias.annotations == target
zachariasTargetDF = CriticalityHelper.getCombinedTopGenes(zachariasDiffTableMap, zacharias.df, 
                        includeCriteria=includeCriteria, missesAllowed=1, minimumChange=2, expressionThreshold=0.01, checkSurface=False)
zachariasTargetDF

In [ ]:
zachariasTargetDF.index

In [ ]:
zachariasTargetDF.loc["Lyz2"]

## Projections

In [ ]:
zacharias.project(NM, "Negretti-Montoro")
# zacharias.project(NMFull, "Negretti-Montoro Full")
# zacharias.project(NMM.basis, "Negretti-Montoro-McCall 3000 0.1")
# zacharias.project(NMP.basis, "Negretti + Montoro + Planer")
# zacharias.project(planer.combinedBases["NM"], "NMP")
# zacharias.project(riemondy.basis, "Riemondy")
# zacharias.project(planer.basis, "Niethamer")
# zacharias.project(bibek.basis, "Bibek")
# zacharias.project(choi.combinedBases["Riemondy"], "CR")
# zacharias.getOrthologs()
# zacharias.project(Natri, "Natri")
# zacharias.project(Natri2000, "Natri2000")
# zacharias.project(KathiriyaABI2, "KathiriyaABI2")
# zacharias.project(KathiriyaABI1, "KathiriyaABI1")
# zacharias.project(Kathiriya3000, "Kathiriya")
# zacharias.project(KathiriyaRelabeled, "KathiriyaRelabeled")

In [ ]:
# zacharias.projections["Natri2000"]
Natri2000

In [ ]:
zacharias.annotations[includeCriteria].value_counts()

In [ ]:
# processedNames = [name.split("_")[1] for name in zacharias.processed.columns]
# set(processedNames)
# YFPNames = [name.split("_")[0] for name in YFP.columns]
YFP = YFP.set_axis(YFPNames, axis=1)
# common = [name for name in processedNames[:1000] if name in YFPNames[:1000]]

In [ ]:
# # zacharias.processed.columns.index = "cell"
# zackProcessed = zacharias.processed
# zackProcessed = zackProcessed.set_axis([name.split("_")[2] for name in zacharias.processed.columns], axis=1)
# zackProcessed
zacharias.processed

In [ ]:
# YFP = pd.read_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/ZachariasYFP.csv", index_col="cell").T
# YFPNames = [name.split("-")[0] + "-1" for name in YFP.columns]
# zacharias.processed = pd.concat([zacharias.processed, YFP], join="inner")
# pd.merge(zackProcessed, YFP, on= how="inner")
# zacharias.processed = zacharias.processed
# set(YFPNames)
del YFP
del zackProcessed

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] < 5000, zacharias.metadata["percent_mito"] < 25)
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] > 250, includeCriteria)
zachariasSimilarityMap = SimilarityHelper.getMatchingProjections(zacharias, "Negretti-Montoro", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(zachariasSimilarityMap, 
        title="Zacharias Projected On Negretti-Montoro", source="PRC2", labels=["AT2", "Basal", "AT1", "Ciliated", "Secretory"],
        labelFontSize=20, outFile="../../PendingResults/Zacharias Reannotated vs NM Boxplot.svg"
)

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] < 5000, zacharias.metadata["percent_mito"] < 25)
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] > 250, includeCriteria)
zachariasSimilarityMap = SimilarityHelper.getMatchingProjections(zacharias, "Kathiriya3000", includeCriteria=includeCriteria)
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
SimilarityHelper.similarityBoxplot(zachariasSimilarityMap, 
        title="Zacharias Projected On Kathiriya", testKeep=labelOrder, #source="PRC2", labels=["AT2", "Krt5", "AT1", "Ciliated", "Secretory", "Alveolar_transitional"],
        labelFontSize=20, outFile="../../PendingResults/Zacharias vs Kathiriya3000 Boxplot.png"
)

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] < 5000, zacharias.metadata["percent_mito"] < 25)
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] > 500, includeCriteria)
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
ax = SimilarityHelper.plotTwo(zacharias, 'Negretti-Montoro', 'AT1', 'AT2',
                         includeCriteria=includeCriteria, unsupervisedContour=False, source="PRC2", alpha=1, labels=labelOrder,
                         markerSize=60, legendMarkerScale=2.75, axisFontSize=20, legendFontSize=18,
                         DPI=300, supervisedContour=False, gene="YFP", #maxLabelCount=1000,
                         # title="Zacharias Projected Onto NM Reference", 
                         outFile="../../PendingResults/Zacharias Reannotated vs NM AT1 vs AT2 YFP.svg",
)

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] < 5000, zacharias.metadata["percent_mito"] < 25)
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] > 500, includeCriteria)
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
ax = SimilarityHelper.plotTwo(zacharias, 'Niethamer', 'Alveolar_transitional', 'AT2',
                         includeCriteria=includeCriteria, unsupervisedContour=False, source="PRC2", alpha=1, labels=labelOrder,
                         markerSize=60, legendMarkerScale=2.75, axisFontSize=20, legendFontSize=18,
                         DPI=300, supervisedContour=False, maxLabelCount=1000,
                         # outFile="../../PendingResults/Zacharias Reannotated vs Planer 0.1 Basal vs AT2.svg",
)

In [ ]:
includeCriteria = zacharias.metadata[zacharias.timeColumn].isin(["2mo_KO", "4mo_KO", "9mo_KO"])
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
# timeOrder = ["2mo_Ctrl", "4mo_Ctrl", "9mo_Ctrl", "2mo_KO", "4mo_KO", "9mo_KO"]
ax, samples = SimilarityHelper.plotTwo(zacharias, 'Natri', 'Basal', 'AT2',
    labels=labelOrder, source="PRC2", includeCriteria=includeCriteria, maxLabelCount=None, getSamples=True,
    DPI=300, # markerSize=60, legendMarkerScale=2.75, axisFontSize=20, legendFontSize=18,
    outFile='../../PendingResults/Zacharias vs Natri Basal vs AT2 Only KO.svg'
)

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata[zacharias.timeColumn].isin(["2mo_KO", "4mo_KO", "9mo_KO"]), zacharias.annotations.index.isin(samples))
ax = SimilarityHelper.plotTwo(zacharias, 'Natri', 'KRT5-KRT17+', 'Basal',
    alternateAnnotations=zacharias.metadata[zacharias.timeColumn], #labels=labelOrder,
    source="ggplot2", includeCriteria=includeCriteria, maxLabelCount=None,
    markerSize=60, legendMarkerScale=2.75, axisFontSize=20, legendFontSize=18, DPI=300,
    outFile='../../PendingResults/Zacharias vs Natri SKAR vs Basal Only KO Time Points Downsampled 500.png'
)

In [ ]:
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] < 5000, zacharias.metadata["percent_mito"] < 25)
includeCriteria = np.logical_and(zacharias.metadata["nFeature_RNA"] > 500, includeCriteria)
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
ax = SimilarityHelper.plotTwo(zacharias, 'KathiriyaABI2', 'Basal', 'ABI2',
                         includeCriteria=includeCriteria, unsupervisedContour=False, alpha=1, labels=labelOrder, source="PRC2",
                         markerSize=60, legendMarkerScale=2.75, axisFontSize=20, legendFontSize=18,# figX=8, figY=8,
                         DPI=300, supervisedContour=False, maxLabelCount=None,
                         outFile="../../PendingResults/Zacharias vs KathiriyaABI2 Basal vs ABI2.png",
)

In [ ]:
SimilarityHelper.getProjectionStats(zacharias, "Natri", "AT1", onlyQuantiles=True, outFile="../../PendingResults/Zacharias vs Natri AT1 Quantile Projections.png")

In [ ]:
SimilarityHelper.plotThree(zacharias, "Kathiriya", "ABI1", "AEC2s", "Basal", maxLabelCount=500)

In [ ]:
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
SimilarityHelper.plotProjectionResultsMatrix(zacharias, "Natri", labels=labelOrder, cbar=True, square=True, outFile="../../PendingResults/Zacharias vs Natri Projections Matrix.png")

In [ ]:
from scipy import stats
t_stats, p_values = stats.ttest_1samp(zacharias.projections["Natri"].loc[:, zacharias.annotations == "ABI"], popmean=0.2, axis=1, alternative="greater")

# 3. Apply FDR Correction (Benjamini-Hochberg procedure)
fdr_p_values = stats.false_discovery_control(p_values, method='bh')
fdr_p_values

In [ ]:
SimilarityHelper.getOverThresholdSignificanceMatrix(zacharias, "Natri", threshold=0.2, test="pvalue",
                                                    outFile="../../PendingResults/Zacharias vs Natri T-Test Projection Pvalues.png"
)

In [ ]:
# fig, ax = plt.subplots(2, 2, figsize=(20, 20))
fig, ax = plt.subplots(2, 3, figsize=(35, 20))
# marker_genes = ['Sftpc', 'Sftpa1', 'Sftpd', 'Cbr2', 'Nkx2-1', 'Fabp5'] # AT2
# marker_genes = ['Mki67', 'Cdk1', 'Cenpa', 'Top2a'] # Proliferation
# marker_genes = ['Lcn2', 'Il33', 'Retnla', 'Etv5', 'Abca3', 'Cebpa'] # Activated/Primed AT2
# marker_genes = ['Hopx', 'Pdpn', 'Cav1', 'Vegfa', 'Ager', 'Rtkn2'] # AT1
# marker_genes = ['Ndrg1', 'Sprr1a', 'Cldn4', 'Clu', 'Ly6a', 'Sfn'] # DATP
marker_genes = ['Krt5', 'Krt15', 'Trp63', 'Aqp3', 'Ngfr', 'Dapl1'] # Basal

axs = ax.flatten()
for i, gene in enumerate(marker_genes):
    _ = SimilarityHelper.plotTwo(zacharias, "Negretti-Plasschaert", 'AT1', 'AT2',
            gene=gene, ax=axs[i], show=False, #maxLabelCount=500,
            # includeCriteria=zacharias.metadata[zacharias.timeColumn] == "Day_0"
)
plt.savefig('../PendingResults/Zacharias AT1 vs AT2 Basal Markers.png')
plt.show()

In [ ]:
# includeCriteria = ["KO" in val for val in zacharias.metadata[zacharias.timeColumn]]
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
timeOrder = ["2mo_Ctrl", "4mo_Ctrl", "9mo_Ctrl", "2mo_KO", "4mo_KO", "9mo_KO"]
SimilarityHelper.plotTwoMultiple(zacharias, 'Niethamer', 'AT1', 'AT2',
    labels=labelOrder, subsetNames=timeOrder, source="PRC2", gene="YFP",
    alpha=1, markerSize=150, legendMarkerScale=0.25, axisFontSize=48, legendFontSize=48, DPI=300,
    outFile='../PendingResults/Zacharias Reannotated vs Planer All Days AT1 vs AT2 YFP.svg'
)

In [ ]:
#includeCriteria = ["KO" in val for val in zacharias.metadata[zacharias.timeColumn]]
labelOrder = ["AT2", "Basal-like", "KO_AT2", "ABI", "AT1"]
timeOrder = ["2mo_Ctrl", "4mo_Ctrl", "9mo_Ctrl", "2mo_KO", "4mo_KO", "9mo_KO"]
ax = SimilarityHelper.plotTwoMultiple(zacharias, 'Natri', 'KRT5-KRT17+', 'AT2',
    labels=labelOrder, subsetNames=timeOrder, source="PRC2",
    alpha=1, markerSize=150, legendMarkerScale=0.25, axisFontSize=48, legendFontSize=48, figX=12, figY=12, DPI=300,
    includeCriteria=None,
    outFile='../../PendingResults/Zacharias vs Natri All Days SKAR vs AT2.svg'
)

In [ ]:
zacharias.setBasis()
dimensionsMap = Perturbation.getSpaceDimensionsAll(zacharias, zacharias.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, outFile="../../PendingResults/Zacharias Perturbation Dimensions.png")

# McCall

In [ ]:
mcCall = TopObject.TopObject("McCall", skipProcess=True)

In [ ]:
mcCall.annotations.value_counts()

In [ ]:
mcCallCopy = mcCall.copy()
includeCriteria = mcCallCopy.annotations.isin(["AT1", "AT2", "Alveolar intermediate", "Basal-like", "MCC"])
mcCallCopy.filter(condition=includeCriteria, maxSamples=755)
# mcCall.filterBestGenes(0.2)
mcCallCopy.setBasis()
# mcCall.combineBases(NM, firstKeep="Alveolar intermediate", name="NMM")

In [ ]:
includeCriteria=mcCall.annotations.isin(["AT1", "AT2", "Alveolar intermediate", "Early intermediate", "Basal-like", "MCC"])
mcCall.testBasis(maxBasisSamples=240, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(mcCall, title="McCall Basis Confusion Matrix", decimalMode="Clean",
                                              outFile="../../PendingResults/McCall Confusion Matrix Downsampled Basis 240 Test 500 Trials 5.png"
)

In [ ]:
# mcCall.project(planer.basis, "Planer")
# mcCall.project(NMFull, "Negretti-Montoro")
# mcCall.project(zacharias.combinedBases["ZNM"], "Negretti-Montoro-Zacharias")
# mcCall.project(planer.combinedBases["NMSmall"], "Negretti-Montoro-Planer")
# mcCall.getOrthologs(mapping, inplace=True)
# mcCall.project(HaberMAP500, "HaberMAP500")
# mcCall.project(HaberMAP, "HaberMAP")
mcCall.project(Adams, "Adams")

In [ ]:
mcCallSimilarityMap = SimilarityHelper.getMatchingProjections(mcCall, "Negretti-Montoro", includeCriteria=None)
SimilarityHelper.similarityBoxplot(mcCallSimilarityMap, 
        title="McCall Projected Onto Negretti-Montoro Reference",
        # outFile="../../PendingResults/McCall vs Negretti-Montoro Full Boxplot.png"
)

In [ ]:
mcCallSimilarityMap = SimilarityHelper.getMatchingProjections(mcCall, "Negretti-Montoro-Planer", includeCriteria=None)
SimilarityHelper.similarityBoxplot(mcCallSimilarityMap, 
        title="McCall Projected Onto Negretti-Montoro-Planer Reference",
        outFile="../../PendingResults/McCall vs NMP Small Boxplot.png"
)

In [ ]:
includeCriteria=mcCall.annotations.isin(["AT1", "AT2", "Alveolar intermediate", "Early intermediate", "Basal-like"])
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(mcCall, "Adams", "Aberrant_Basaloid", "Basal",
                         includeCriteria=includeCriteria, 
                         maxLabelCount=None, #unsupervisedContour=True, 
                         title="McCall Projected Onto Adams Reference", 
                         outFile="../../PendingResults/McCall vs Adams 500 ANOVA2 SKAR vs Basal.png",
)

In [ ]:
includeCriteria=mcCall2.annotations.isin(["AT1", "AT2", "Alveolar intermediate", "Early intermediate", "Basal-like", "BASC-like", "Mucous secretory", "Secretory"])
ax = SimilarityHelper.plotTwoMultiple(mcCall, "Negretti-Montoro Full", "Secretory", "AT2",
                         subsetCategory=mcCall.metadata["orig.ident"], subsetNames=list(set(mcCall.metadata["orig.ident"])),
                         includeCriteria=includeCriteria, xBounds=(-0.2, 0.5), yBounds=(-0.2, 0.9),
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="McCall Projected Onto Negretti-Montoro Reference", 
                         outFile="../../PendingResults/McCall vs Negretti-Montoro Full Secretory vs AT2 All Treatments.png",
)

In [ ]:
mcCall.metadata["orig.ident"].value_counts()

In [ ]:
includeCriteria=mcCall.annotations.isin(["AT1", "AT2", "Alveolar intermediate", "Early intermediate", "Basal-like"])
marker_genes = ['Krt5', 'Krt15', 'Trp63', 'Aqp3', 'Ngfr', 'Dapl1'] # Basal
SimilarityHelper.plotMultipleGenes(mcCall, "Negretti-Montoro", 'Basal', 'AT2', marker_genes, includeCriteria=includeCriteria)

In [ ]:
mcCall.setBasis()
res = Perturbation.runProcessedSpaceExperiment(mcCall, mcCall.basis, "AT2", "AT1", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT2 → AT1", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(mcCall, mcCall.basis, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
mcCall.setBasis()
mcCallDimensionsMap = Perturbation.getSpaceDimensionsAll(mcCall, mcCall.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False, cellTypes=["AT1", "AT2", "Alveolar intermediate", "Early intermediate", "Basal-like", "MCC"])
SimilarityHelper.plotSpaceMatrix(mcCallDimensionsMap, outFile="../../PendingResults/McCall Perturbation Dimensions.png")

# Konkimalla

In [ ]:
konkimallaBHT = TopObject.TopObject("KonkimallaBHT", skipProcess=True)
konkimallaBHT.metadata

In [ ]:
konkimallaBHT.annotations.value_counts()

In [ ]:
konkimallaBHT.annotations[np.logical_and(konkimallaBHT.metadata["condition"] == "BHT", konkimallaBHT.annotations.isin(konkimallaBHT.toKeep))].value_counts()

In [ ]:
# konkimallaBHT.project(NM, "Negretti-Montoro")
# konkimallaBHT.project(NMFull, "Negretti-Montoro Full")
# konkimallaBHT.project(mcCall.basis, "McCall 755")
# konkimallaBHT.project(planer.basis, "Planer 0.2")
# konkimallaBHTOrtho = konkimallaBHT.copy()
konkimallaBHT.getOrthologs(mapping, inplace=True)
konkimallaBHT.project(HaberMAP500, "HaberMAP500")
konkimallaBHT.project(HaberMAP, "HaberMAP")

In [ ]:
konkimallaSimilarityMap = SimilarityHelper.getMatchingProjections(konkimallaBHT, "Negretti-Montoro Full", testKeep=konkimallaBHT.toKeep)
SimilarityHelper.similarityBoxplot(konkimallaSimilarityMap, title="Konkimalla vs Negretti-Montoro Similarity Boxplot", 
                                   # outFile="../../PendingResults/Konkimalla vs Negretti-Montoro Boxplot.png"
)

In [ ]:
konkimallaSimilarityMap = SimilarityHelper.getMatchingProjections(konkimallaBHT, "Planer 276", testKeep=konkimallaBHT.toKeep)
SimilarityHelper.similarityBoxplot(konkimallaSimilarityMap, title="Konkimalla vs Planer 276 Similarity Boxplot", 
                                   # outFile="../../PendingResults/Konkimalla vs Planer 276 Boxplot.png"
)

In [ ]:
includeCriteria=konkimallaBHT.annotations.isin(konkimallaBHT.toKeep)
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(konkimallaBHT, "Negretti-Montoro", "AT1", "AT2",
                         includeCriteria=includeCriteria, 
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="Konkimalla BHT Projected Onto Negretti-Montoro Reference", 
                         # outFile="../../PendingResults/Konkimalla BHT vs Negretti-Montoro AT1 vs AT2.png",
)

In [ ]:
markerGenes = ['Ndrg1', 'Sprr1a', 'Cldn4', 'Clu', 'Ly6a', 'Sfn', 'Krt19', 'Krt8']
includeCriteria=konkimallaBHT.annotations.isin(konkimallaBHT.toKeep)
SimilarityHelper.plotMultipleGenes(konkimallaBHT, "Planer 276", 'Alveolar_transitional', 'AT2', markerGenes, 
                                   includeCriteria=includeCriteria, outFile="../../PendingResults/KonkimallaBHT vs Planer 276 ABI Markers.png"
)

In [ ]:
includeCriteria=np.logical_and(konkimallaBHT.annotations.isin(konkimallaBHT.toKeep), konkimallaBHT.annotations != "Secretory") # Secretory not in McCall basis
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(konkimallaBHT, "McCall 755", "Alveolar intermediate", "AT2",
                         includeCriteria=includeCriteria, 
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="Konkimalla BHT Projected Onto McCall Reference", 
                         # outFile="../../PendingResults/Konkimalla BHT vs McCall 755 AT1 vs AT2.png",
)

In [ ]:
includeCriteria=konkimallaBHT.annotations.isin(konkimallaBHT.toKeep)
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(konkimallaBHT, "Planer 276", "Alveolar_transitional", "AT2",
                         includeCriteria=includeCriteria, 
                         # maxLabelCount=500, unsupervisedContour=True, 
                         title="Konkimalla BHT Projected Onto Planer 276 Reference", 
                         outFile="../../PendingResults/Konkimalla BHT vs Planer 276 Alveolar_transitional vs AT2.png",
)

In [ ]:
includeCriteria=konkimallaBHTOrtho.annotations.isin(konkimallaBHTOrtho.toKeep)
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(konkimallaBHTOrtho, "HaberMAP", "KRT5-/KRT17+", "AT2",
                         includeCriteria=includeCriteria, 
                         # unsupervisedContour=True, maxLabelCount=500,
                         title="Konkimalla BHT Projected Onto HaberMAP Reference", 
                         outFile="../../PendingResults/Konkimalla BHT vs HaberMAP SKAR vs AT2.png",
)

In [ ]:
SimilarityHelper.getProjectionStats(konkimallaBHT, "HaberMAP", "ABI", target="KRT5-/KRT17+",
                                    # outFile="../../PendingResults/Konkimalla BHT vs HaberMAP Regenerative_transitional Stats.csv",
)

In [ ]:
SimilarityHelper.getProjectionStatsFocused(konkimallaBHT, "HaberMAP", "KRT5-/KRT17+",
                                    # outFile="../../PendingResults/Konkimalla BHT vs HaberMAP Regenerative_transitional Stats.csv",
)

In [ ]:
alphas = np.linspace(0, 1.0, 31)
# src_ids = konkimallaBHT.df.columns[konkimallaBHT.annotations.isin(konkimallaBHT.toKeep)]
src_ids = konkimallaBHT.df.columns[konkimallaBHT.annotations == "AT2"]
res = Perturbation.run_processed_space_experiment(
    adata=konkimallaBHT.anndata,
    basis=NM,
    source_cell_ids=src_ids,
    source_label="AT2",
    target_label="AT1",
    alphas=alphas,
    max_cells=1000,
    random_state=1,
    filter_to_correct_at_alpha0=False,
    verbose=True,
)

Perturbation.plot_flip_curves(res, "AT2 → AT1", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
alphas = np.linspace(0, 1.0, 31)
res = Perturbation.runProcessedSpaceExperiment(konkimallaBHT, NM, "AT2", "AT1", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)

Perturbation.plot_flip_curves(res, "AT2 → AT1", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(konkimallaBHT, NM, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # output_file='SCGB3A2_to_AT1.png'
)

## AT1 Ablation

In [ ]:
konkimallaAT1Ablation = TopObject.TopObject("KonkimallaAT1Ablation", skipProcess=True)
konkimallaAT1Ablation.metadata

In [ ]:
konkimallaAT1Ablation.annotations.value_counts()

In [ ]:
konkimallaAT1Ablation.metadata["condition"].value_counts()

In [ ]:
konkimallaAT1Ablation.annotations[konkimallaAT1Ablation.metadata["condition"] == "BHT"].value_counts()

In [ ]:
konkimallaAT1Ablation.annotations[~konkimallaAT1Ablation.metadata["condition"].isin(["Control10X", "Homeo", "BHT"])].value_counts()

In [ ]:
includeCriteria=konkimalla.annotations.isin(["AT1", "AT2", "PATS"])
konkimalla.testBasis(maxBasisSamples=400, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)#, allowedGenes=planer.df.index)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(konkimalla, title="Konkimalla Basis Confusion Matrix", decimalMode="Clean",
                                              # outFile="../../PendingResults/Konkimalla Ablation Confusion Matrix Downsampled Basis 500 Test 500 Trials 5.png"
)

In [ ]:
konkimallaAT1Ablation.project(NM, "Negretti-Montoro")
konkimallaAT1Ablation.project(NMFull, "Negretti-Montoro Full")
# # konkimallaBHT.project(mcCall.basis, "McCall 755")
# # konkimallaBHT.project(planer.basis, "Planer 0.2")
# konkimallaBHTOrtho = konkimallaBHT.copy()
# konkimallaBHTOrtho.getOrthologs(mapping, inplace=True)
# konkimallaBHTOrtho.project(HaberMAP500, "HaberMAP500")
# konkimallaBHTOrtho.project(HaberMAP, "HaberMAP")

In [ ]:
includeCriteria = ~konkimallaAT1Ablation.metadata["condition"].isin(["Control10X", "Homeo", "BHT"])
ax = SimilarityHelper.plotTwo(konkimallaAT1Ablation, "Negretti-Montoro", "AT1", "AT2",
                         includeCriteria=includeCriteria, gene="Trp53",
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="Konkimalla AT1 Ablation Projected Onto Negretti-Montoro Reference", 
                         # outFile="../../PendingResults/Konkimalla AT1 Ablation vs NM AT1 vs AT2.png",
)

In [ ]:
includeCriteria=konkimallaAT1Ablation.metadata["condition"] == "BHT"
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwo(konkimallaAT1Ablation, "Negretti-Montoro", "AT1", "AT2",
                         includeCriteria=includeCriteria, gene="Mdm2",
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="Konkimalla BHT Projected Onto Negretti-Montoro Reference", 
                         # outFile="../../PendingResults/Konkimalla BHT vs NM AT1 vs AT2.png",
)

In [ ]:
# markerGenes = ['Ndrg1', 'Sprr1a', 'Cldn4', 'Clu', 'Ly6a', 'Sfn', 'Krt19', 'Krt8'] # SKAR
# markerGenes = ['Sftpc', 'Sftpa1', 'Sftpd', 'Cbr2', 'Nkx2-1', 'Fabp5'] # AT2
# markerGenes = ['Mki67', 'Cdk1', 'Cenpa', 'Top2a'] # Proliferation
# markerGenes = ['Lcn2', 'Il33', 'Retnla', 'Etv5', 'Abca3', 'Cebpa'] # Activated/Primed AT2
markerGenes = ['Hopx', 'Pdpn', 'Cav1', 'Vegfa', 'Ager', 'Rtkn2'] # AT1
includeCriteria = ~konkimallaAT1Ablation.metadata["condition"].isin(["Control10X", "Homeo", "BHT"])
SimilarityHelper.plotMultipleGenes(konkimallaAT1Ablation, "Negretti-Montoro", 'AT1', 'AT2', markerGenes, 
                                   includeCriteria=includeCriteria,
                                   # outFile="../../PendingResults/Konkimalla AT1 Ablation vs NM AT1 Markers.png"
)

In [ ]:
markerGenes = ['Ndrg1', 'Sprr1a', 'Cldn4', 'Clu', 'Ly6a', 'Sfn', 'Krt19', 'Krt8'] # SKAR
# markerGenes = ['Sftpc', 'Sftpa1', 'Sftpd', 'Cbr2', 'Nkx2-1', 'Fabp5'] # AT2
# markerGenes = ['Mki67', 'Cdk1', 'Cenpa', 'Top2a'] # Proliferation
# markerGenes = ['Lcn2', 'Il33', 'Retnla', 'Etv5', 'Abca3', 'Cebpa'] # Activated/Primed AT2
# markerGenes = ['Hopx', 'Pdpn', 'Cav1', 'Vegfa', 'Ager', 'Rtkn2'] # AT1

SimilarityHelper.plotMultipleGenes(konkimallaAT1Ablation, "Negretti-Montoro", 'AT1', 'AT2', markerGenes, 
                                   includeCriteria=konkimallaAT1Ablation.metadata["condition"] == "BHT", 
                                   outFile="../../PendingResults/Konkimalla BHT vs NM SKAR Markers.png"
)

In [ ]:
# includeCriteria=konkimallaAT1Ablation.annotations.isin(konkimallaAT1Ablation.toKeep)
# includeCriteria = np.logical_and(includeCriteria, mcCall.metadata["orig.ident"] == "SD Bleo")
ax = SimilarityHelper.plotTwoMultiple(konkimallaAT1Ablation, "Negretti-Montoro", "AT1", "AT2", DPI=300, gene="Trp53",
                         # includeCriteria=konkimallaAT1Ablation.metadata["condition"] == "BHT", 
                         #maxLabelCount=1000, #unsupervisedContour=True, 
                         title="Konkimalla All Conditions Projected Onto Negretti-Montoro Reference", 
                         outFile="../../PendingResults/Konkimalla All Conditions vs NM Trp53 Expression.png",
)

In [ ]:
# includeCriteria=konkimalla.annotations.isin(["AT1", "PATS", "Activated AT2"])
# konkimalla.setBasis(includeCriteria=includeCriteria, maxSamples=500)
# konkimallaDimensionsMap = Perturbation.getSpaceDimensionsAll(konkimalla, konkimalla.basis, maxCells=1000, randomState=7, filterToCorrectAtAlpha0=False, cellTypes=None)
SimilarityHelper.plotSpaceMatrix(konkimallaDimensionsMap, figX=8, figY=8, title="Konkimalla Perturbations",
                                 outFile="../../PendingResults/Konkimalla All Perturbation Dimensions.png"
)

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(konkimallaAT1Ablation, konkimallaAT1Ablation.basis, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # outFile="../../PendingResults/Konkimalla Perturbation AT1 -> AT2.png"
)

# Tang

In [ ]:
tang = TopObject.TopObject("Tang", skipProcess=True)
tang.metadata

In [ ]:
Tang7 = SimilarityHelper.rawToAnnData("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/Tang/Day7_ExpressionMatrix/matrix.mtx", "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/Tang/Day7_ExpressionMatrix/features.tsv", None, geneHeader=None)
# Tang7
names = pd.read_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/Tang/Day7_ExpressionMatrix/barcodes.tsv", sep="\t", header=None)
names[0]

In [ ]:
T7 = TopObject.TopObject("Tang7", annObject=Tang7, cellTypeColumn="Sample", skipProcess=True)
T7.metadata

In [ ]:
# tang.project(NM, "Negretti-Montoro")
# tang.project(NMFull, "Negretti-Montoro")
# tang.project(McCall.basis, "McCall")
tang.getOrthologs(None, inplace=True)
tang.project(Natri, "Natri") 

In [ ]:
# newAnno = [val if val != "N7" else "C7" for val in tang.annotations]
# tang.anndata.obs["Sample"] = newAnno
# tang.setMetadata()
tang.metadata

In [ ]:
tangSimilarityMap = SimilarityHelper.getMatchingProjections(tang, "Natri")
SimilarityHelper.similarityBoxplot(tangSimilarityMap,
                                   title="Tang vs Natri", outFile="../../PendingResults/Tang vs Natri Boxplot.png"
)

In [ ]:
ax = SimilarityHelper.plotTwo(tang, "Negretti-Montoro", "AT1", "AT2",
                         includeCriteria=None, #alternateAnnotations=tang.metadata["Sample"],
                         # maxLabelCount=500, #unsupervisedContour=True, 
                         title="Tang Projected Onto Negretti-Montoro Reference", 
                         outFile="../../PendingResults/Tang vs NM Full AT1 vs AT2.png",
)

In [ ]:
ax = SimilarityHelper.plotTwo(tang, "Natri", "KRT5-KRT17+", "AT2",
                         includeCriteria=None,
                         maxLabelCount=500, #unsupervisedContour=True, 
                         title="Tang Projected Onto Natri Reference", 
                         outFile="../../PendingResults/Tang vs Natri SKAR vs AT2 Downsampled 500.png",
)

# Overlays

In [ ]:
## Load TopObjects
strunz = TopObject.TopObject("Strunz", skipProcess=False, keep=True)
riemondy = TopObject.TopObject("Riemondy", skipProcess=False)
choi = TopObject.TopObject("Choi", skipProcess=False)
planer = TopObject.TopObject("Planer", skipProcess=False)
mcCall = TopObject.TopObject("McCall", skipProcess=False)
auyeung = TopObject.TopObject("Auyeung", skipProcess=False)
# bibek = TopObject.TopObject("Bibek", skipProcess=False)
kobayashi = TopObject.TopObject("Kobayashi", skipProcess=False)
konkimalla = TopObject.TopObject("KonkimallaAT1Ablation", skipProcess=False)
planer.name = "Niethamer"
konkimalla.name = "Konkimalla"

In [ ]:
choi.getOrthologs(mapping, inplace=True)
riemondy.getOrthologs(mapping, inplace=True)
planer.getOrthologs(mapping, inplace=True)
mcCall.getOrthologs(mapping, inplace=True)
strunz.getOrthologs(mapping, inplace=True)
auyeung.getOrthologs(mapping, inplace=True)
konkimalla.getOrthologs(mapping, inplace=True)

In [ ]:
## Overlay different datasets
includeCriteriaList = []
includeCriteriaList.append(choi.annotations.isin(["Intermediate"]))
includeCriteriaList.append(riemondy.annotations.isin(["Cell Cycle Arrest Type II", "Transdifferentiating Type II"]))
includeCriteriaList.append(planer.annotations.isin(["Alveolar_transitional", "AT2", "AT1", "AT1_AT2"]))
includeCriteriaList.append(mcCall.annotations.isin(["Alveolar intermediate", "Early intermediate"]))
includeCriteriaList.append(strunz.annotations.isin(["Krt8+ ADI"]))
includeCriteriaList.append(auyeung.annotations.isin(["Fibrotic_transitional", "Regenerative_transitional"]))
# includeCriteriaList.append(kobayashi.annotations.isin(["Intermediate"]))
includeCriteriaList.append(konkimalla.annotations.isin(["PATS"]))

# topObjects = [kaminski2020, habermann, bharat, PPFE]
topObjects = [choi, riemondy, planer, mcCall, strunz, auyeung, konkimalla]
# otherProjections, otherAnnotations, names = SimilarityHelper.setupOverlay(topObjects, "LungMAP", includeCriteriaList, basis=lungMAP, forceProject=False)
otherProjections, otherAnnotations, names = SimilarityHelper.setupOverlay(topObjects, "LungMAP", includeCriteriaList, basis=lungMAP2500, forceProject=True)

In [ ]:
## axis1 = "Krt8+ ADI"
# axis1 = "KRT5-/KRT17+"
axis1 = "AT1"
axis2 = "AT2"
overlayLabels = ["Niethamer AT2", "Niethamer AT1", "Niethamer AT1_AT2", "Niethamer Alveolar_transitional", "Choi Intermediate", "Riemondy Cell Cycle Arrest Type II", "Riemondy Transdifferentiating Type II", "Strunz Krt8+ ADI", "Auyeung Fibrotic_transitional", "Auyeung Regenerative_transitional", "Konkimalla PATS"]
for i in range(len(overlayLabels)):
    ax = SimilarityHelper.plotTwo(choi, "HaberMAP", axis1, axis2,
                 additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
                 includeCriteria=includeCriteriaList[0], overlayLabels=overlayLabels[:i+1], labelDimensions=True, unsupervisedContour=False, maxLabelCount=500,
                 title="Choi, Riemondy, Niethamer, McCall, Strunz, Auyeung, and Konkimalla Projected on LungMAP", axisFontSize=18, legendFontSize=13, legendInner=True, titleFontSize=15, DPI=300,
                 # outFile="../../Results/Overlays/Overlay Choi, Riemondy, Niethamer, Strunz, Auyeung, and Konkimalla vs LungMAP 500 ANOVA2 AT1 vs AT2 Downsampled 500 Intermediates (" + str(i) + ").png",
    )

In [ ]:
topObject = konkimallaAT1Ablation
basis = "McCall"
celltype = "PATS"
target = "Alveolar intermediate"
SimilarityHelper.getProjectionStats(topObject, basis, celltype, target=target,
                                    outFile="../../PendingResults/Konkimalla vs " + basis + " 755 " + celltype + " Stats.csv",
)
SimilarityHelper.getProjectionStatsFocused(topObject, basis, target,
                                            outFile="../../PendingResults/Konkimalla vs " + basis + " 755 Stats 2.csv"
)